# ĐÁNH GIÁ MÔ HÌNH RAG LEVEL 2 VỚI KHO TRI THỨC V2 (ĐA SEED 42, 44, 46) — COCO 2014

Notebook này thực hiện **Zero-shot Inference / Ablation Benchmark** cho mô hình **Level 2 (`RAGBaselineLevel2`)** khi kết nối với **Kho tri thức V2 (`knowledge_base_level4`)** trên **5.000 ảnh COCO Karpathy Test** qua 3 Seeds (`42, 44, 46`).

### Các nâng cấp và sửa lỗi quan trọng trong phiên bản này:
1. **Khai thác trực tiếp trường ngữ nghĩa của Kho tri thức V2 (`kb_v2_dedup_object_grounded`)**:
   - Loại bỏ triệt để các câu trùng lặp trong kho bằng trường `duplicate_of` và khóa chuẩn hóa NFKC của KB V2 (tránh việc 5 câu trùng nhau chiếm hết Top-5 khi tính trung bình có trọng số).
   - Ưu tiên các câu có thực thể `objects` đã được spaCy trích xuất trong KB V2 và thưởng nhẹ mật độ cấu trúc ngữ nghĩa (`objects + attributes + actions + relations`), trong khi vẫn giữ nguyên thang điểm tương đồng cosine CLIP để đưa vào bộ hòa trộn `Weighted-Sum Fusion` ($lpha = 0.5$).
2. **Sửa lỗi nạp nhầm Checkpoint giữa các Seed**:
   - Tuyệt đối không dùng fallback `seed42` cho `seed 44` hoặc `seed 46`. Mỗi seed chỉ nạp đúng checkpoint của chính seed đó và in mã băm `SHA-256` của `state_dict` để xác minh 3 mô hình độc lập.
3. **Kiểm tra nghiêm ngặt Trọng số Mô hình (`Strict Key Validation`)**:
   - Tự động ánh xạ chuẩn xác cả định dạng checkpoint có tiền tố `decoder.*` (`RAGBaselineLevel2`) lẫn định dạng phẳng (`BaselineCaptioner`) và bắt buộc `missing_keys == []` trên toàn bộ tầng Giải mã.
4. **Đồng bộ Bộ tham chiếu Chuẩn Karpathy Test (25.000 câu) & Giao thức Java METEOR/SPICE**:
   - Tự động chuẩn hóa đúng 5 câu tham chiếu/ảnh cho 5.000 ảnh Test (tương thích 100% với Level 5 v3) và tải sẵn `stanford-corenlp-3.6.0` qua HTTPS.


In [1]:
# ==============================================================================
# CELL 1 — CÀI ĐẶT THƯ VIỆN, THIẾT LẬP JAVA SẠCH STDOUT & KIỂM TRA METEOR/SPICE
# ==============================================================================
import os, sys, re, shutil, subprocess, tempfile, tarfile, urllib.request, zipfile, math
from pathlib import Path

os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

# 1. Gỡ bỏ các biến môi trường Java có thể chèn log lạ vào stdout
for _env_name in ("JAVA_TOOL_OPTIONS", "_JAVA_OPTIONS", "JDK_JAVA_OPTIONS"):
    if os.environ.pop(_env_name, None) is not None:
        print(f"Đã gỡ biến môi trường {_env_name} để bảo vệ giao thức Java METEOR/PTBTokenizer.")

from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
else:
    print("✅ Google Drive đã được mount tại /content/drive.")

RUNTIME_DIR = Path("/content/coco_eval_runtime")
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "faiss-cpu", "pycocoevalcap==1.2", "transformers>=4.45,<5",
    "nltk>=3.9,<4", "tqdm>=4.66,<5", "scipy", "psutil"
])

import nltk
for _pkg in ("punkt", "punkt_tab", "wordnet", "omw-1.4"):
    nltk.download(_pkg, quiet=True)

# 2. Cài đặt Java 11 & tạo Wrapper tắt hoàn toàn cảnh báo Cgroup trên stdout của Colab
# (Nguyên nhân chính gây lệch dòng PTBTokenizer làm CIDEr tụt xuống ~3.92 và lỗi METEOR b'11.0 10.0...')
def _is_java11(java_path):
    try:
        res = subprocess.run([str(java_path), "-version"], capture_output=True, text=True, timeout=15)
        return res.returncode == 0 and 'version "11.' in (res.stderr + res.stdout)
    except Exception:
        return False

java_candidates = list(Path("/usr/lib/jvm").glob("java-11-*/bin/java")) + list((RUNTIME_DIR / "java11").glob("*/bin/java"))
if not any(_is_java11(p) for p in java_candidates):
    try:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-11-jre-headless"], check=True)
    except Exception:
        java_root = RUNTIME_DIR / "java11"
        java_root.mkdir(exist_ok=True)
        with tempfile.TemporaryDirectory(dir=RUNTIME_DIR) as tmp:
            archive = Path(tmp) / "jdk11.tar.gz"
            urllib.request.urlretrieve(
                "https://api.adoptium.net/v3/binary/latest/11/ga/linux/x64/jre/hotspot/normal/eclipse",
                archive,
            )
            with tarfile.open(archive, "r:gz") as tar:
                tar.extractall(java_root)
    java_candidates = list(Path("/usr/lib/jvm").glob("java-11-*/bin/java")) + list((RUNTIME_DIR / "java11").glob("*/bin/java"))

REAL_JAVA_BIN = next((p for p in java_candidates if _is_java11(p)), None) or Path(shutil.which("java"))
assert REAL_JAVA_BIN is not None and Path(REAL_JAVA_BIN).is_file(), "❌ Không tìm thấy Java!"

# Tạo wrapper /content/coco_eval_runtime/clean_java_bin/java luôn truyền cờ -XX:-UseContainerSupport -Xlog:os+container=off
CLEAN_JAVA_DIR = RUNTIME_DIR / "clean_java_bin"
CLEAN_JAVA_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_JAVA_BIN = CLEAN_JAVA_DIR / "java"
CLEAN_JAVA_BIN.write_text(
    f"#!/usr/bin/env bash\nexec \"{REAL_JAVA_BIN}\" -XX:-UseContainerSupport -Xlog:os+container=off \"$@\"\n",
    encoding="utf-8",
)
CLEAN_JAVA_BIN.chmod(0o755)

os.environ["JAVA_HOME"] = str(Path(REAL_JAVA_BIN).resolve().parent.parent)
os.environ["PATH"] = f"{CLEAN_JAVA_DIR}:{Path(REAL_JAVA_BIN).resolve().parent}:{os.environ.get('PATH', '')}"
java_bin = shutil.which("java")
print("Java active:", subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0])

# 3. Chuẩn bị Stanford CoreNLP 3.6.0 cho SPICE & giới hạn Heap -Xmx4G để không bị tràn 12.7 GB RAM Colab T4
import pycocoevalcap.spice.spice as spice_module
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer, PUNCTUATIONS
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.spice.spice import Spice
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

spice_dir = Path(spice_module.__file__).parent
spice_lib = spice_dir / "lib"
spice_lib.mkdir(parents=True, exist_ok=True)
for jar_name in ("stanford-corenlp-3.6.0.jar", "stanford-corenlp-3.6.0-models.jar"):
    jar_path = spice_lib / jar_name
    if not jar_path.is_file() or not zipfile.is_zipfile(jar_path):
        url = f"https://repo.maven.apache.org/maven2/edu/stanford/nlp/stanford-corenlp/3.6.0/{jar_name}"
        tmp_jar = RUNTIME_DIR / (jar_name + ".tmp")
        print(f"⏳ Đang tải tài nguyên SPICE: {jar_name}...")
        urllib.request.urlretrieve(url, tmp_jar)
        assert zipfile.is_zipfile(tmp_jar), f"❌ File jar tải về bị hỏng: {jar_name}"
        shutil.copyfile(tmp_jar, jar_path)
        tmp_jar.unlink(missing_ok=True)

# Đặt thư mục tạm và cache của SPICE trên ổ SSD cục bộ /content (tránh lỗi quyền ghi và tràn bộ nhớ)
spice_module.TEMP_DIR = str(RUNTIME_DIR / "spice_tmp")
spice_module.CACHE_DIR = str(RUNTIME_DIR / "spice_cache")
Path(spice_module.TEMP_DIR).mkdir(parents=True, exist_ok=True)
Path(spice_module.CACHE_DIR).mkdir(parents=True, exist_ok=True)

# Patch -Xmx8G -> -Xmx4G trong spice.py để không bị Linux OOM Killer giết trên Colab T4 (12.7 GB RAM)
_spice_py_path = Path(spice_module.__file__)
_spice_src = _spice_py_path.read_text(encoding="utf-8")
if "'-Xmx8G'" in _spice_src:
    _spice_py_path.write_text(_spice_src.replace("'-Xmx8G'", "'-Xmx4G'"), encoding="utf-8")
    import importlib
    importlib.reload(spice_module)
    from pycocoevalcap.spice.spice import Spice
    spice_module.TEMP_DIR = str(RUNTIME_DIR / "spice_tmp")
    spice_module.CACHE_DIR = str(RUNTIME_DIR / "spice_cache")
    print("✅ Đã tối ưu bộ nhớ SPICE (-Xmx4G) phù hợp với 12.7 GB RAM của Colab T4.")

# 4. Định nghĩa SafePTBTokenizer & ManagedMeteor (chống 100% lỗi lệch dòng do ký tự \r hoặc cảnh báo JVM)
class SafePTBTokenizer:
    """Bọc PTBTokenizer chuẩn của pycocoevalcap nhưng:
    1) Làm sạch triệt để \r, \n, \t trong mọi câu trước khi gửi vào Java.
    2) Lọc bỏ mọi dòng cảnh báo JVM ([warning][os,container]...) nếu có trên stdout.
    3) Kiểm tra nghiêm ngặt số dòng trả về đúng bằng số câu đầu vào (không bao giờ lệch image_id)."""
    _JVM_WARN_RE = re.compile(r"^\[\d+[.,]\d+s\]\[(?:warning|info|error)\]|Cgroup memory controller", re.IGNORECASE)

    def tokenize(self, captions_for_image):
        cmd = [
            "java", "-XX:-UseContainerSupport", "-Xlog:os+container=off",
            "-cp", "stanford-corenlp-3.4.1.jar",
            "edu.stanford.nlp.process.PTBTokenizer",
            "-preserveLines", "-lowerCase",
        ]
        final_tokenized = {}
        image_ids = []
        clean_sents = []
        for k, v in captions_for_image.items():
            for c in v:
                image_ids.append(k)
                txt = re.sub(r"[\r\n\t]+", " ", str(c["caption"])).strip()
                clean_sents.append(txt if txt else "empty")

        sentences = "\n".join(clean_sents)
        tok_dir = Path( sys.modules["pycocoevalcap.tokenizer.ptbtokenizer"].__file__ ).parent
        p_tokenizer = subprocess.Popen(
            cmd, cwd=str(tok_dir),
            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL
        )
        out_bytes, _ = p_tokenizer.communicate(input=sentences.encode("utf-8"))
        raw_lines = out_bytes.decode("utf-8", errors="replace").split("\n")
        if raw_lines and raw_lines[-1] == "":
            raw_lines.pop()

        lines = [ln for ln in raw_lines if not self._JVM_WARN_RE.search(ln)]
        assert len(lines) == len(image_ids), (
            f"❌ Lỗi lệch số dòng PTBTokenizer: nhận {len(lines)} dòng cho {len(image_ids)} câu!"
        )
        for k, line in zip(image_ids, lines):
            if k not in final_tokenized:
                final_tokenized[k] = []
            tok_cap = " ".join([w for w in line.rstrip().split(" ") if w not in PUNCTUATIONS])
            final_tokenized[k].append(tok_cap)
        return final_tokenized

class ManagedMeteor(Meteor):
    def close(self):
        process = getattr(self, "meteor_p", None)
        if process is not None and process.poll() is None:
            try:
                if process.stdin is not None:
                    process.stdin.close()
            except Exception:
                pass
            finally:
                try:
                    process.kill()
                    process.wait(timeout=15)
                except Exception:
                    pass
    def __del__(self):
        self.close()

# 5. Smoke Test ngay tại Cell 1 để đảm bảo PTBTokenizer, METEOR và SPICE đạt đúng 1.0000
_smoke_tk = SafePTBTokenizer()
_smoke_gts = _smoke_tk.tokenize({1: [{"caption": "a cat sits on a mat"}]})
_smoke_res = _smoke_tk.tokenize({1: [{"caption": "a cat sits on a mat"}]})
assert _smoke_gts[1][0] == "a cat sits on a mat", f"❌ PTBTokenizer bị nhiễm stdout lạ: {_smoke_gts}"

_m_scorer = ManagedMeteor()
try:
    _m_val, _ = _m_scorer.compute_score(_smoke_gts, _smoke_res)
finally:
    _m_scorer.close()
_sp_val, _ = Spice().compute_score(_smoke_gts, _smoke_res)
assert math.isfinite(float(_m_val)) and float(_m_val) > 0.99, f"❌ Smoke test METEOR thất bại: {_m_val}"
assert math.isfinite(float(_sp_val)) and float(_sp_val) > 0.99, f"❌ Smoke test SPICE thất bại: {_sp_val}"

import numpy as np
import torch
import faiss
print(f"✅ Kiểm tra bộ chấm điểm hoàn tất: PTBTokenizer OK | METEOR={_m_val:.4f} | SPICE={_sp_val:.4f}")
print(f"Torch: {torch.__version__} | FAISS: {faiss.__version__} | Java: {java_bin}")


Mounted at /content/drive
Java active: openjdk version "11.0.32.1" 2026-08-18
⏳ Đang tải tài nguyên SPICE: stanford-corenlp-3.6.0.jar...
⏳ Đang tải tài nguyên SPICE: stanford-corenlp-3.6.0-models.jar...
✅ Đã tối ưu bộ nhớ SPICE (-Xmx4G) phù hợp với 12.7 GB RAM của Colab T4.
✅ Kiểm tra bộ chấm điểm hoàn tất: PTBTokenizer OK | METEOR=1.0000 | SPICE=1.0000
Torch: 2.11.0+cu130 | FAISS: 1.15.1 | Java: /content/coco_eval_runtime/clean_java_bin/java


In [2]:
# ==============================================================================
# CELL 2 — CẤU HÌNH ĐƯỜNG DẪN, DÒ TÌM CHECKPOINT 3 SEEDS (42, 44, 46) & CHUẨN HÓA TEST REFS
# ==============================================================================
import os, sys, json, time, hashlib, re
from pathlib import Path
import numpy as np
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị sử dụng: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# 1. Xác định DATA_ROOT
DATA_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/DataKhoaLuan/Data"),
    Path("/content/drive/.shortcut-targets-by-id/1OCkkcQKlIYTCPLnGgb2y4aTf_VJFE7xQ/Data"),
    Path("/content/drive/MyDrive/Data/Data"),
    Path("/content/drive/MyDrive/Data"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.is_dir()), None)
assert DATA_ROOT is not None, "❌ Không tìm thấy thư mục DATA_ROOT trên Google Drive!"
print(f"DATA_ROOT: {DATA_ROOT}")

# 2. Đường dẫn Kho tri thức V2 (knowledge_base_level4)
KB_ROOT_CANDIDATES = [
    DATA_ROOT / "knowledge_base_level4",
    Path("/content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level4"),
    DATA_ROOT / "khotrithuccaitien",
]
KB_DIR = next((p for p in KB_ROOT_CANDIDATES if p.is_dir()), None)
assert KB_DIR is not None, f"❌ Không tìm thấy thư mục Kho tri thức V2 tại {KB_ROOT_CANDIDATES[0]}!"
print(f"KB_DIR (Kho tri thức V2): {KB_DIR}")

# 3. Cấu hình SEEDS cần đánh giá và đường dẫn Checkpoint riêng cho từng Seed (KHÔNG fallback chéo seed!)
SEEDS_TO_EVAL = [42, 44, 46]  #@param {type:"raw"}

# Nếu bạn để checkpoint ở thư mục riêng, có thể điền trực tiếp đường dẫn vào đây:
CHECKPOINT_OVERRIDES = {
    42: None,
    44: None,
    46: None,
}

def find_level2_checkpoint_for_seed(seed):
    override = CHECKPOINT_OVERRIDES.get(seed)
    if override and Path(override).is_file():
        return Path(override)

    level2_dir = DATA_ROOT / "baseline_level2"
    candidates = [
        level2_dir / f"seed{seed}/checkpoint_best.pt",
        level2_dir / f"seed_{seed}/checkpoint_best.pt",
        level2_dir / f"seed{seed}/checkpoint_best_seed{seed}.pt",
        level2_dir / f"seed_{seed}/checkpoint_best_seed{seed}.pt",
        level2_dir / f"seed{seed}/best_model.pt",
        level2_dir / f"seed_{seed}/best_model.pt",
        level2_dir / f"rag_outputs/R1_K5_N50_A0.5_seed{seed}_L4/checkpoint_best.pt",
        level2_dir / f"rag_outputs/R1_K5_N50_A0.5_seed{seed}/checkpoint_best.pt",
        level2_dir / f"rag_level2_v6/r1_weighted_sum_k5_n50_a0p5_seed{seed}/checkpoint_best.pt",
        DATA_ROOT / f"rag_baseline_level2/R1_K5_N50_A0.5_seed{seed}/checkpoints/checkpoint_best.pt",
        DATA_ROOT / f"rag_outputs/R1_K5_N50_A0.5_seed{seed}_L4/checkpoint_best.pt",
    ]
    # Quét các run trong multiseed_beam3_v1 (nơi Level_2_COCO2014_MultiSeed_BeamSearch_k3.ipynb lưu seed_44, seed_46)
    ms_dir = level2_dir / "multiseed_beam3_v1"
    if ms_dir.is_dir():
        for run_sub in sorted(ms_dir.glob("run_*")):
            candidates.extend([
                run_sub / f"seed_{seed}/checkpoint_best_seed{seed}.pt",
                run_sub / f"seed_{seed}/checkpoint_best.pt",
                run_sub / f"seed{seed}/checkpoint_best_seed{seed}.pt",
            ])
    for c in candidates:
        if c.is_file():
            return c

    # Tìm kiếm đệ quy có kiểm soát trong baseline_level2 nhưng BẮT BUỘC tên đường dẫn phải chứa đúng số seed
    if level2_dir.is_dir():
        seed_pat = re.compile(rf"(?:seed[_-]?{seed})(?!\d)", re.IGNORECASE)
        for p in sorted(level2_dir.rglob("*.pt")):
            if "last" not in p.name.lower() and "smoke" not in str(p).lower() and seed_pat.search(str(p)):
                return p
    return None

CHECKPOINT_PATHS = {}
missing_seeds = []
for s in SEEDS_TO_EVAL:
    cp = find_level2_checkpoint_for_seed(s)
    if cp is not None:
        CHECKPOINT_PATHS[s] = cp
        print(f"✅ Checkpoint Level 2 (Seed {s}): {cp}")
    else:
        missing_seeds.append(s)
        print(f"⚠️ Chưa tìm thấy checkpoint riêng cho Seed {s} (tuyệt đối không mượn checkpoint của Seed 42).")

assert CHECKPOINT_PATHS, f"❌ Không tìm thấy bất kỳ checkpoint Level 2 nào cho các seed {SEEDS_TO_EVAL}!"

# 4. Tìm Vocab & CLIP Test Features
VOCAB_CANDIDATES = [
    DATA_ROOT / "baseline_level1/vocab.json",
    DATA_ROOT / "baseline_level1/checkpoints/vocab.json",
    DATA_ROOT / "vocab.json",
]
VOCAB_PATH = next((p for p in VOCAB_CANDIDATES if p.is_file()), None)
assert VOCAB_PATH is not None, "❌ Không tìm thấy vocab.json!"
print(f"VOCAB_PATH: {VOCAB_PATH}")

FEATURE_CANDIDATES = [
    DATA_ROOT / "baseline_level1/clip_image_features/test_features.npy",
    DATA_ROOT / "clip_image_features/test_features.npy",
]
FEATURE_PATH = next((p for p in FEATURE_CANDIDATES if p.is_file()), None)
FEATURE_IDS_PATH = FEATURE_PATH.parent / "test_image_ids.json" if FEATURE_PATH else None
assert FEATURE_PATH is not None and FEATURE_IDS_PATH.is_file(), "❌ Không tìm thấy test_features.npy hoặc test_image_ids.json!"
print(f"FEATURE_PATH: {FEATURE_PATH}")

# 5. Nạp & Làm sạch Ground-Truth COCO Karpathy Test (Đồng bộ 100% với bộ chấm chuẩn 5.000 ảnh / 25.010 câu tham chiếu)
import unicodedata
from collections import Counter, defaultdict

ANN_CANDIDATES = [
    DATA_ROOT / "COCO_2014/annotations_trainval2014/annotations/captions_karpathy_test.json",
    DATA_ROOT / "COCO_2014/annotations/captions_karpathy_test.json",
    Path("/content/drive/MyDrive/Data/Giai_nen_coco2014/COCO_2014/annotations/captions_karpathy_test.json"),
]
RAW_ANN_PATH = next((p for p in ANN_CANDIDATES if p.is_file()), None)
assert RAW_ANN_PATH is not None, "❌ Không tìm thấy captions_karpathy_test.json!"

def clean_coco_caption(text):
    """Chuẩn hóa NFKC và gộp mọi ký tự xuống dòng \\r, \\n, \\t thành khoảng trắng đơn (tránh vỡ dòng trong PTBTokenizer/METEOR)."""
    text = unicodedata.normalize("NFKC", "" if text is None else str(text))
    return re.sub(r"\s+", " ", text).strip()

raw_doc = json.loads(RAW_ANN_PATH.read_text(encoding="utf-8"))
clean_anns = []
by_img = defaultdict(list)
for ann in raw_doc.get("annotations", []):
    c_clean = clean_coco_caption(ann.get("caption", ""))
    if c_clean:
        iid = int(ann["image_id"])
        row_clean = {"id": ann.get("id", len(clean_anns) + 1), "image_id": iid, "caption": c_clean}
        clean_anns.append(row_clean)
        by_img[iid].append(row_clean)

assert len(by_img) == 5000, f"❌ Số lượng ảnh trong {RAW_ANN_PATH.name} là {len(by_img)} (cần 5.000 ảnh)!"
ANN_PATH = RAW_ANN_PATH
ANN_DOC_NORMALIZED = {"images": raw_doc.get("images", []), "annotations": clean_anns}
print(f"✅ Đã nạp & làm sạch Ground-Truth Karpathy Test từ {ANN_PATH.name}: {len(by_img):,} ảnh | {len(clean_anns):,} câu tham chiếu.")


Thiết bị sử dụng: cuda (Tesla T4)
DATA_ROOT: /content/drive/MyDrive/DataKhoaLuan/Data
KB_DIR (Kho tri thức V2): /content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level4
✅ Checkpoint Level 2 (Seed 42): /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level2/rag_outputs/R1_K5_N50_A0.5_seed42_L4/checkpoint_best.pt
✅ Checkpoint Level 2 (Seed 44): /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level2/multiseed_beam3_v1/run_3aace7f5829acedd/seed_44/checkpoint_best_seed44.pt
✅ Checkpoint Level 2 (Seed 46): /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level2/multiseed_beam3_v1/run_3aace7f5829acedd/seed_46/checkpoint_best_seed46.pt
VOCAB_PATH: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/vocab.json
FEATURE_PATH: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/clip_image_features/test_features.npy
✅ Đã nạp & làm sạch Ground-Truth Karpathy Test từ captions_karpathy_test.json: 5,000 ảnh | 25,010 câu tham chiếu.


In [3]:
# ==============================================================================
# CELL 3 — NẠP KHO TRI THỨC V2 (FAISS + EMBEDDINGS + MAPPING + CHỈ MỤC NGỮ NGHĨA KB V2)
# ==============================================================================
import pickle
import faiss
from collections import Counter, defaultdict
from nltk.stem import WordNetLemmatizer

EXPECT_FULL_KB = True   # True: bắt buộc kho FULL (566.747 caption)

def resolve_kb_v2_bundle(kb_root):
    """Dò tìm bundle hoàn chỉnh của Kho tri thức V2 (knowledge_base_level4)."""
    kb_root = Path(kb_root)
    candidates = [kb_root]
    if (kb_root / "shared/bundles").is_dir():
        candidates.extend(sorted([p for p in (kb_root / "shared/bundles").iterdir() if p.is_dir()]))
    if (kb_root / "runs").is_dir():
        candidates.extend(sorted(
            [p for p in (kb_root / "runs").iterdir() if p.is_dir()],
            key=lambda p: (not p.name.startswith("coco2014_full"), p.name),
        ))

    def _first(base, rels):
        return next((base / r for r in rels if (base / r).is_file()), None)

    for cand in candidates:
        emb = _first(cand, ["text_embeddings_train_restval.npy", "embeddings/text_embeddings_train_restval.npy"])
        mapping = _first(cand, ["text_embedding_mapping.jsonl", "mapping/text_embedding_mapping.jsonl", "knowledge_base.jsonl"])
        index = _first(cand, ["text_embeddings_train_restval.faiss", "index/text_embeddings_train_restval.faiss"])
        meta = _first(cand, ["metadata.json"])
        img_objs = _first(cand, ["image_objects.jsonl", "semantic_metadata/image_objects.jsonl"])
        if emb is not None and mapping is not None:
            return {"root": cand, "emb": emb, "mapping": mapping, "index": index, "meta": meta, "img_objs": img_objs}
    raise FileNotFoundError(f"❌ Không tìm thấy bundle Kho tri thức V2 hợp lệ dưới {kb_root}!")

KB_BUNDLE = resolve_kb_v2_bundle(KB_DIR)
EMB_FILE     = KB_BUNDLE["emb"]
INDEX_FILE   = KB_BUNDLE["index"]
MAPPING_FILE = KB_BUNDLE["mapping"]
META_FILE    = KB_BUNDLE["meta"]
IMG_OBJ_FILE = KB_BUNDLE["img_objs"]

for name, p in [("EMB", EMB_FILE), ("INDEX", INDEX_FILE), ("MAPPING", MAPPING_FILE), ("META", META_FILE), ("IMG_OBJ", IMG_OBJ_FILE)]:
    print(f"{name:<8}: {p}")

meta = json.loads(META_FILE.read_text(encoding="utf-8")) if META_FILE else {}
print("metadata:", {k: meta.get(k) for k in ("caption_count", "dataset", "model_name", "embedding_dim", "vectors_ready")})
assert meta.get("vectors_ready", True), "❌ metadata.json báo vectors_ready=False: kho chưa build xong vector!"

kb_embeddings = np.load(str(EMB_FILE), mmap_mode="r")
N_KB, D_KB = kb_embeddings.shape
print(f"Tổng số tri thức: {N_KB:,} vectors ({D_KB} chiều)")
if EXPECT_FULL_KB:
    assert N_KB == 566_747, f"❌ Kho có {N_KB:,} vector (cần 566.747 vector của tập train+restval)."

if INDEX_FILE is not None and INDEX_FILE.is_file():
    print("Đang nạp FAISS IndexFlatIP từ file...")
    kb_index = faiss.read_index(str(INDEX_FILE))
else:
    print("Đang dựng FAISS IndexFlatIP trong RAM...")
    kb_index = faiss.IndexFlatIP(D_KB)
    for _s in range(0, N_KB, 16384):
        kb_index.add(np.ascontiguousarray(kb_embeddings[_s:_s + 16384], dtype=np.float32))
assert kb_index.ntotal == N_KB, f"❌ FAISS ({kb_index.ntotal}) lệch với embeddings ({N_KB})"
print(f"✅ FAISS Index sẵn sàng với {kb_index.ntotal:,} vectors.")

# Bảng chuẩn hóa đồng nghĩa danh từ (nhất quán với cấu hình đồ án)
_LEMMATIZER = WordNetLemmatizer()
_NOUN_ALIASES = {
    "bike": "bicycle", "bicycle": "bicycle", "bicycles": "bicycle", "bikes": "bicycle",
    "man": "person", "woman": "person", "child": "person", "boy": "person", "girl": "person",
    "people": "person", "person": "person", "men": "person", "women": "person", "kid": "person",
    "plane": "airplane", "airplane": "airplane", "aeroplane": "airplane", "jet": "airplane",
    "couch": "sofa", "sofa": "sofa", "tv": "television", "cellphone": "phone",
}
_TOKEN_RE = re.compile(r"[a-z]+(?:'[a-z]+)?", flags=re.IGNORECASE)

def canonicalize_concept(word):
    w = str(word).strip().lower()
    if not w:
        return ""
    tokens = _TOKEN_RE.findall(w)
    if not tokens:
        return w
    head = _LEMMATIZER.lemmatize(tokens[-1], pos="n")
    return _NOUN_ALIASES.get(head, head)

def load_or_build_kb_v2_compact_index(mapping_path, img_obj_path, expected_n):
    """Đọc text_embedding_mapping.jsonl (và image_objects.jsonl) của KB V2, trích xuất:
    - captions: list[str]
    - image_ids: int64[N]
    - offsets: int64[N] (để seek đọc chi tiết JSON của từng dòng khi cần)
    - duplicate_of: int32[N] (-1 nếu là câu gốc canonical, >=0 nếu là câu trùng lặp trong KB V2)
    - obj_sets: list[frozenset[str]] (tập nhãn thực thể objects đã chuẩn hóa từ KB V2 + COCO instances)
    - rel_sets: list[frozenset[tuple]] (tập bộ ba SPO relations, actions, attributes từ KB V2)
    - triple_counts: int16[N] (tổng số lượng cấu trúc ngữ nghĩa của mỗi caption)
    Lưu cache nhị phân để những lần chạy sau chỉ mất ~2 giây."""
    stat = mapping_path.stat()
    cache_path = KB_DIR / f"kb_v2_compact_semantics_{expected_n}_{stat.st_size}.pkl"
    if cache_path.is_file():
        try:
            t0 = time.time()
            with open(cache_path, "rb") as f:
                data = pickle.load(f)
            if len(data["captions"]) == expected_n:
                print(f"♻️ Đã nạp nhanh Chỉ mục Ngữ nghĩa KB V2 từ cache ({time.time()-t0:.1f}s): {cache_path.name}")
                return data
        except Exception as e:
            print(f"⚠️ Cache ngữ nghĩa cũ không đọc được ({e}), sẽ xây lại từ JSONL...")

    t0 = time.time()
    # 1. Nạp thêm nhãn vật thể từ image_objects.jsonl (nếu có)
    img_ann_objs = {}
    if img_obj_path is not None and img_obj_path.is_file():
        with open(img_obj_path, "rb") as f:
            for line in f:
                if not line.strip():
                    continue
                row = json.loads(line)
                ikey = row.get("image_key")
                labels = {canonicalize_concept(o.get("label", "")) for o in row.get("objects", []) if o.get("label")}
                labels.discard("")
                if ikey and labels:
                    img_ann_objs[ikey] = frozenset(labels)

    captions = []
    image_ids = np.empty(expected_n, dtype=np.int64)
    offsets = np.empty(expected_n, dtype=np.int64)
    duplicate_of = np.full(expected_n, -1, dtype=np.int32)
    triple_counts = np.zeros(expected_n, dtype=np.int16)
    obj_sets = []
    rel_sets = []

    off = 0
    idx = 0
    with open(mapping_path, "rb") as f:
        for line in f:
            line_len = len(line)
            if line.strip():
                row = json.loads(line)
                rid = int(row.get("row_id", idx))
                assert rid == idx, f"❌ row_id lệch thứ tự tại dòng {idx} (row_id={rid})"
                captions.append(row.get("caption", ""))
                image_ids[idx] = int(row["image_id"])
                offsets[idx] = off

                dup = row.get("duplicate_of")
                if dup is not None:
                    duplicate_of[idx] = int(dup)

                # Trích xuất Objects & Triples (Attributes, Actions, Relations) từ KB V2
                raw_objs = row.get("objects", [])
                id2label = {}
                c_objs = set()
                for o in raw_objs:
                    oid = o.get("id")
                    lbl = canonicalize_concept(o.get("label") or o.get("name") or "")
                    if lbl:
                        c_objs.add(lbl)
                        if oid:
                            id2label[oid] = lbl

                # Bổ sung ngữ cảnh vật thể cấp ảnh (image_objects_ref) nếu có
                iref = row.get("image_objects_ref")
                if iref and iref in img_ann_objs:
                    c_objs.update(img_ann_objs[iref])

                c_rels = set()
                for a in row.get("attributes", []):
                    s_lbl = id2label.get(a.get("subject"), canonicalize_concept(a.get("subject", "")))
                    val = str(a.get("value", "")).strip().lower()
                    if s_lbl and val:
                        c_rels.add(("attr", s_lbl, val))
                for ac in row.get("actions", []):
                    s_lbl = id2label.get(ac.get("subject"), canonicalize_concept(ac.get("subject", "")))
                    pred = _LEMMATIZER.lemmatize(str(ac.get("predicate", "")).strip().lower(), pos="v")
                    if s_lbl and pred:
                        c_rels.add(("act", s_lbl, pred))
                for r in row.get("relations", []):
                    s_lbl = id2label.get(r.get("subject"), canonicalize_concept(r.get("subject", "")))
                    o_lbl = id2label.get(r.get("object"), canonicalize_concept(r.get("object", "")))
                    pred = str(r.get("predicate", "")).strip().lower()
                    if s_lbl and o_lbl and pred:
                        c_rels.add(("rel", s_lbl, pred, o_lbl))

                obj_sets.append(frozenset(c_objs))
                rel_sets.append(frozenset(c_rels))
                triple_counts[idx] = min(32000, len(c_rels))
                idx += 1
            off += line_len

    assert idx == expected_n, f"❌ Số dòng đọc được ({idx}) khác N_KB ({expected_n})"
    data = {
        "captions": captions,
        "image_ids": image_ids,
        "offsets": offsets,
        "duplicate_of": duplicate_of,
        "obj_sets": obj_sets,
        "rel_sets": rel_sets,
        "triple_counts": triple_counts,
    }
    try:
        tmp_c = cache_path.with_suffix(".pkl.tmp")
        with open(tmp_c, "wb") as fw:
            pickle.dump(data, fw, protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(tmp_c, cache_path)
        print(f"💾 Đã lưu cache Chỉ mục Ngữ nghĩa KB V2 tại: {cache_path.name}")
    except Exception as e:
        print(f"ℹ️ Không ghi được cache phụ trợ lên Drive ({e}), tiếp tục dùng trong RAM.")
    print(f"✅ Đã nạp {expected_n:,} dòng ngữ nghĩa KB V2 trong {time.time()-t0:.1f}s.")
    return data

KB_V2_DATA = load_or_build_kb_v2_compact_index(MAPPING_FILE, IMG_OBJ_FILE, N_KB)
kb_captions      = KB_V2_DATA["captions"]
kb_image_ids     = KB_V2_DATA["image_ids"]
kb_offsets       = KB_V2_DATA["offsets"]
kb_duplicate_of  = KB_V2_DATA["duplicate_of"]
kb_obj_sets      = KB_V2_DATA["obj_sets"]
kb_rel_sets      = KB_V2_DATA["rel_sets"]
kb_triple_counts = KB_V2_DATA["triple_counts"]

n_dups = int((kb_duplicate_of >= 0).sum())
n_with_objs = sum(1 for s in kb_obj_sets if len(s) > 0)
n_with_rels = int((kb_triple_counts > 0).sum())
print(f"📊 Thống kê KB V2: {n_dups:,} câu trùng lặp (duplicate_of) | {n_with_objs:,} câu có Objects | {n_with_rels:,} câu có Triples (Attr/Act/Rel).")

def fetch_semantic(row_id):
    """Đọc đầy đủ 1 dòng JSON từ text_embedding_mapping.jsonl theo byte offset."""
    with open(MAPPING_FILE, "rb") as f:
        f.seek(int(kb_offsets[row_id]))
        return json.loads(f.readline())


EMB     : /content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level4/text_embeddings_train_restval.npy
INDEX   : /content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level4/text_embeddings_train_restval.faiss
MAPPING : /content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level4/text_embedding_mapping.jsonl
META    : /content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level4/metadata.json
IMG_OBJ : /content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level4/image_objects.jsonl
metadata: {'caption_count': 566747, 'dataset': 'coco2014', 'model_name': 'openai/clip-vit-base-patch32', 'embedding_dim': 512, 'vectors_ready': True}
Tổng số tri thức: 566,747 vectors (512 chiều)
Đang nạp FAISS IndexFlatIP từ file...
✅ FAISS Index sẵn sàng với 566,747 vectors.
♻️ Đã nạp nhanh Chỉ mục Ngữ nghĩa KB V2 từ cache (4.2s): kb_v2_compact_semantics_566747_2698984528.pkl
📊 Thống kê KB V2: 21,916 câu trùng lặp (duplicate_of) | 566,671 câu có Objects | 521,602 câu có Triples (Attr/Act/Rel).


In [4]:
# ==============================================================================
# CELL 4 — KIẾN TRÚC MÔ HÌNH LEVEL 2 (RAGBaselineLevel2) & NẠP TRỌNG SỐ NGHIÊM NGẶT
# ==============================================================================
from torch import nn
import torch.nn.functional as F

with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    vocab = json.load(f)
stoi = {str(k): int(v) for k, v in vocab["stoi"].items()}
itos = {int(v): str(k) for k, v in stoi.items()}
VOCAB_SIZE = len(stoi)
PAD_ID, BOS_ID, EOS_ID, UNK_ID = (stoi.get(t, i) for i, t in enumerate(("<PAD>", "<BOS>", "<EOS>", "<UNK>")))
MAX_LEN = 30

class BaselineCaptioner(nn.Module):
    def __init__(self, clip_dim=512, vocab_size=VOCAB_SIZE, d_model=256, nhead=8, num_layers=3, dropout=0.1, max_length=30):
        super().__init__()
        self.d_model = d_model
        self.max_length = max_length
        self.feature_projection = nn.Linear(clip_dim, d_model)
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.position_embedding = nn.Embedding(max_length, d_model)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
            norm_first=False,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.output_projection = nn.Linear(d_model, vocab_size)

    @staticmethod
    def causal_mask(length, device):
        return torch.triu(torch.ones(length, length, dtype=torch.bool, device=device), diagonal=1)

    def forward(self, image_features, input_ids):
        batch_size, seq_len = input_ids.shape
        memory = self.feature_projection(image_features).unsqueeze(1)
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
        target = self.token_embedding(input_ids) + self.position_embedding(positions)
        decoded = self.decoder(
            target,
            memory,
            tgt_mask=self.causal_mask(seq_len, input_ids.device),
            tgt_key_padding_mask=input_ids.eq(PAD_ID),
        )
        return self.output_projection(decoded)

class RAGBaselineLevel2(nn.Module):
    def __init__(self, decoder=None, alpha=0.5, temperature=1.0):
        super().__init__()
        self.decoder = decoder if decoder is not None else BaselineCaptioner()
        self.alpha = float(alpha)
        self.temperature = float(temperature)
        self.concat_proj = nn.Linear(512 * 2, 512)

    def fuse(self, visual, knowledge_features=None, retrieval_scores=None, mode="weighted_sum"):
        visual = F.normalize(visual, p=2, dim=-1)
        if mode == "visual_only":
            return visual
        weights = torch.softmax(retrieval_scores / self.temperature, dim=-1)
        knowledge = torch.bmm(weights.unsqueeze(1), knowledge_features).squeeze(1)
        knowledge = F.normalize(knowledge, p=2, dim=-1)
        if mode == "knowledge_only":
            return knowledge
        if mode == "concat_linear":
            return F.normalize(self.concat_proj(torch.cat([visual, knowledge], dim=-1)), p=2, dim=-1)
        return F.normalize(self.alpha * visual + (1.0 - self.alpha) * knowledge, p=2, dim=-1)

    def forward(self, visual, knowledge_features, retrieval_scores, input_ids, fusion_mode="weighted_sum"):
        fused = self.fuse(visual, knowledge_features, retrieval_scores, mode=fusion_mode)
        return self.decoder(fused, input_ids)

def compute_state_sha256(state_dict):
    h = hashlib.sha256()
    for k in sorted(state_dict.keys()):
        v = state_dict[k]
        if torch.is_tensor(v):
            t = v.detach().cpu().contiguous()
            h.update(k.encode("utf-8"))
            h.update(t.view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

def load_level2_model_by_seed(seed, ckpt_path):
    """Nạp mô hình Level 2 và kiểm tra strict 100% các tham số của Decoder."""
    model = RAGBaselineLevel2(BaselineCaptioner(), alpha=0.5, temperature=1.0).to(DEVICE)
    try:
        ckpt = torch.load(str(ckpt_path), map_location="cpu", weights_only=True)
    except Exception:
        ckpt = torch.load(str(ckpt_path), map_location="cpu", weights_only=False)

    raw_state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    state = dict(raw_state)

    # Chuẩn hóa tên khóa nếu checkpoint lưu phẳng (BaselineCaptioner) hoặc dùng tên feature_proj
    if any(k.startswith("decoder.token_embedding.") or k.startswith("decoder.decoder.") for k in state):
        # Đã ở định dạng RAGBaselineLevel2 (decoder.*)
        if "decoder.feature_proj.weight" in state and "decoder.feature_projection.weight" not in state:
            state["decoder.feature_projection.weight"] = state.pop("decoder.feature_proj.weight")
            state["decoder.feature_projection.bias"] = state.pop("decoder.feature_proj.bias")
        res = model.load_state_dict(state, strict=False)
        missing_decoder = [k for k in res.missing_keys if k.startswith("decoder.")]
        assert not missing_decoder and not res.unexpected_keys, (
            f"❌ Checkpoint Seed {seed} lệch kiến trúc! Missing={missing_decoder}, Unexpected={res.unexpected_keys}"
        )
    else:
        if "feature_proj.weight" in state and "feature_projection.weight" not in state:
            state["feature_projection.weight"] = state.pop("feature_proj.weight")
            state["feature_projection.bias"] = state.pop("feature_proj.bias")
        model.decoder.load_state_dict(state, strict=True)

    state_sha = compute_state_sha256(model.decoder.state_dict())
    print(f"✅ Đã nạp strict thành công Level 2 (Seed {seed}) | Decoder SHA-256: {state_sha[:16]}...")
    model.eval()
    return model, {"state_sha256": state_sha}

# Kiểm tra trước tất cả các checkpoint đã tìm thấy
_loaded_hashes = {}
for _s, _p in CHECKPOINT_PATHS.items():
    _m, _meta = load_level2_model_by_seed(_s, _p)
    _loaded_hashes[_s] = _meta["state_sha256"]
    del _m
if len(_loaded_hashes) > 1 and len(set(_loaded_hashes.values())) < len(_loaded_hashes):
    print("⚠️ CẢNH BÁO: Có ít nhất 2 seed đang trỏ tới cùng một trọng số mô hình!")


✅ Đã nạp strict thành công Level 2 (Seed 42) | Decoder SHA-256: fdce5927668ef96a...
✅ Đã nạp strict thành công Level 2 (Seed 44) | Decoder SHA-256: 9b25264198ba5307...
✅ Đã nạp strict thành công Level 2 (Seed 46) | Decoder SHA-256: d25553aa8ac5295f...


In [5]:
# ==============================================================================
# CELL 5 — TRUY HỒI ĐA GÓC NHÌN SPO, ĐỊNH HƯỚNG LOGIT & ĐỒNG THUẬN TF-IDF KHO V2 (LEVEL 2: TOP_K=5)
# ==============================================================================
import csv
import math
from collections import Counter

RETRIEVAL_PROTOCOL_NAME = "level2_kb_v2_spo_consensus_decoding_v3_top5"

# THÔNG SỐ HỌC THUẬT CHUẨN & CẤU HÌNH GIẢI MÃ ĐỊNH HƯỚNG TRI THỨC KHO V2 (METHOD 2):
LEVEL_ID           = 2
TOP_N_CANDIDATES   = 50
TOP_K              = 5
W_CLIP             = 1.0
W_SEMANTIC         = 0.0
USE_MMR_DIVERSITY  = False
MMR_LAMBDA         = 1.0
TAU                = 1.0
BEAM_SIZE          = 3
BEAM_EXPAND_SIZE   = 5   # Số nhánh ứng viên mở rộng để tái xếp hạng Đồng thuận TF-IDF & SPO
BEAM_ALPHA         = 0.6
MAX_LEN            = 30
PROTOTYPE_BETA     = 0.78
PRIMARY_SEED_W     = 0.68        # Trọng số nhánh Seed chính trong hợp nhất đa góc nhìn Kho V2
SPO_LOGIT_WEIGHT   = 0.24        # Độ mạnh cổng thưởng từ vựng SPO & N-gram đồng thuận
CIDER_MBR_WEIGHT   = 0.42        # Trọng số tái xếp hạng Đồng thuận TF-IDF CIDEr trên Kho V2
SPO_MBR_WEIGHT     = 0.18        # Trọng số độ phủ cấu trúc SPO khi chọn câu đầu ra
KB_V2_GATE_BOOST   = 0.0        # Độ mở bổ sung cho cổng GatedFusion khi dùng tri thức sạch KB V2

STOPWORDS_SET = {
    "a", "an", "the", "in", "on", "of", "at", "to", "with", "and", "or", "for",
    "is", "are", "was", "were", "be", "been", "being", "by", "from", "as", "into",
    "this", "that", "these", "those", "it", "its", "their", "his", "her", "there"
}

def build_kb_v2_idf_table():
    """Tính bảng trọng số IDF (1-gram..4-gram) thuần túy từ 566.747 câu Train của Kho V2 (0% đụng tập Test)."""
    if "KB_V2_IDF_CACHE" in globals() and isinstance(globals()["KB_V2_IDF_CACHE"], dict):
        return globals()["KB_V2_IDF_CACHE"]
    t_idf = time.time()
    df_uni = Counter()
    total_docs = len(kb_captions)
    # Lấy mẫu chuẩn xác trên tập KB V2 để tính IDF nhanh (~2 giây)
    step = 2 if total_docs > 200000 else 1
    sampled_docs = 0
    tok_pat = re.compile(r"[a-z0-9]+(?:'[a-z]+)?")
    for cap in kb_captions[::step]:
        toks = set(tok_pat.findall(cap.lower()))
        for w in toks:
            df_uni[w] += 1
        sampled_docs += 1
    idf_uni = {
        w: float(math.log((sampled_docs + 1.0) / (cnt + 1.0)) + 1.0)
        for w, cnt in df_uni.items()
    }
    default_idf = float(math.log(sampled_docs + 1.0) + 1.0)
    globals()["KB_V2_IDF_CACHE"] = {"idf_uni": idf_uni, "default_idf": default_idf}
    print(f"✅ Đã khởi tạo bảng trọng số TF-IDF từ Kho tri thức V2 ({len(idf_uni):,} từ vựng) trong {time.time()-t_idf:.1f}s.")
    return globals()["KB_V2_IDF_CACHE"]

KB_V2_IDF_INFO = build_kb_v2_idf_table()

def load_reltr_test_scene_graphs(reltr_path, rerank_cache_path=None):
    scene_graphs = {}
    if reltr_path is not None and Path(reltr_path).is_file():
        reltr_path = Path(reltr_path)
        if reltr_path.suffix == ".json":
            raw = json.loads(reltr_path.read_text(encoding="utf-8"))
            for k, entry in raw.items():
                iid = int(entry.get("image_id", k))
                objs = {canonicalize_concept(o) for o in entry.get("objects", []) if o}
                objs.discard("")
                triplets = set()
                for t in entry.get("triplets", []):
                    s = canonicalize_concept(t.get("subject", ""))
                    o = canonicalize_concept(t.get("object", ""))
                    r = str(t.get("relation", "")).strip().lower()
                    if s and o and r:
                        triplets.add((s, r, o))
                scene_graphs[iid] = {"objects": objs, "triplets": triplets}
        elif reltr_path.suffix == ".csv":
            img_pat = re.compile(r"(\d{12})(?=\.jpg$)", re.IGNORECASE)
            with reltr_path.open("r", encoding="utf-8-sig", newline="") as f:
                for row in csv.DictReader(f):
                    m = img_pat.search(Path(str(row.get("image", ""))).name)
                    if not m:
                        continue
                    iid = int(m.group(1))
                    entry = scene_graphs.setdefault(iid, {"objects": set(), "triplets": set()})
                    pred = str(row.get("prediction", "")).strip()
                    if pred == "no_relation" or " - " not in pred:
                        continue
                    parts = [p.strip() for p in pred.split(" - ", maxsplit=2)]
                    if len(parts) == 3:
                        s, r, o = canonicalize_concept(parts[0]), parts[1].lower(), canonicalize_concept(parts[2])
                        if s: entry["objects"].add(s)
                        if o: entry["objects"].add(o)
                        if s and r and o: entry["triplets"].add((s, r, o))
        print(f"✅ Đã nạp đồ thị cảnh RelTR cho {len(scene_graphs):,} ảnh Test từ {reltr_path.name}.")

    if rerank_cache_path is not None and Path(rerank_cache_path).is_file():
        with np.load(rerank_cache_path, allow_pickle=False) as z:
            c_ids = [int(x) for x in z["image_ids"]]
            c_rows = np.asarray(z["row_indices"], dtype=np.int64)
            for idx_i, iid in enumerate(c_ids):
                if iid not in scene_graphs or not scene_graphs[iid]["objects"]:
                    obj_c = Counter()
                    rel_c = Counter()
                    for r_id in c_rows[idx_i]:
                        for o in kb_obj_sets[int(r_id)]:
                            obj_c[o] += 1
                        for rel_it in kb_rel_sets[int(r_id)]:
                            rel_c[rel_it] += 1
                    scene_graphs[iid] = {
                        "objects": {o for o, cnt in obj_c.items() if cnt >= 2} or set(obj_c.keys()),
                        "triplets": {(it[1], it[2], it[3]) for it, cnt in rel_c.items() if it[0] == "rel" and cnt >= 1},
                    }
        print(f"✅ Đã kết nối thêm cache tham chiếu cho {len(c_ids):,} ảnh từ {Path(rerank_cache_path).name}.")
    return scene_graphs

RELTR_SCENE_GRAPHS = load_reltr_test_scene_graphs(
    globals().get("RELTR_TEST_PATH", None),
    globals().get("RERANK_CACHE_W0P6_PATH", None),
)

def expand_scene_graph_with_kb_v2(reltr_objs, reltr_triplets, cand_rows):
    """Khối A — Semantic Expansion bằng Đồ thị Ngữ nghĩa KB V2."""
    expanded_objs = set(reltr_objs)
    expanded_rels = {("rel", s, r, o) for (s, r, o) in reltr_triplets}

    obj_votes = Counter()
    rel_votes = Counter()
    top_window = 8 if LEVEL_ID == 2 else 12
    for rank_i, r_nb in enumerate(cand_rows[:top_window]):
        w_vote = 1.0 / (1.0 + 0.15 * rank_i)
        for o in kb_obj_sets[r_nb]:
            obj_votes[o] += w_vote
        for rel_it in kb_rel_sets[r_nb]:
            rel_votes[rel_it] += w_vote

    obj_thresh = 2.4 if LEVEL_ID == 2 else 2.0
    for o, v_score in obj_votes.items():
        if v_score >= obj_thresh or (not expanded_objs and v_score >= 1.4):
            expanded_objs.add(o)
    for rel_it, v_score in rel_votes.items():
        if v_score >= 1.4:
            expanded_rels.add(rel_it)
    return expanded_objs, expanded_rels

def compute_kb_v2_relevance_score(row_id, clip_norm_val, target_objects, target_relations):
    """Khối B — Hybrid Ranking: Tính điểm liên quan chuẩn hóa từ CLIP + Đồ thị SPO KB V2."""
    cand_objs = kb_obj_sets[row_id]
    if not cand_objs or not target_objects:
        return float(clip_norm_val)
    matched_objs = sum(1 for o in cand_objs if o in target_objects)
    precision_o = matched_objs / len(cand_objs)
    recall_o    = matched_objs / min(4, len(target_objects))
    obj_overlap = min(1.0, 0.70 * precision_o + 0.30 * min(1.0, recall_o))

    cand_rels = kb_rel_sets[row_id]
    if not cand_rels:
        sem_sc = float(obj_overlap)
    else:
        rel_hits = 0.0
        for item in cand_rels:
            if item in target_relations:
                rel_hits += 1.0
            elif item[0] == "rel":
                _, s, p, o = item
                if s in target_objects and o in target_objects:
                    rel_hits += 0.65
            else:
                _, s, val = item
                if s in target_objects:
                    rel_hits += 0.55
        sem_sc = float(min(1.0, 0.75 * obj_overlap + 0.25 * min(1.0, rel_hits / len(cand_rels))))

    if LEVEL_ID == 2:
        return float(0.82 * clip_norm_val + 0.18 * sem_sc)
    return float(W_CLIP * clip_norm_val + W_SEMANTIC * sem_sc)

def _kb_v2_pair_similarity(rid_a, vec_a, rid_b, vec_b):
    """Khối B — MMR (Diversity): Độ tương đồng lai kết hợp Cosine Vector + Jaccard Thực thể & Quan hệ SPO."""
    cos_sim = max(0.0, float(np.dot(vec_a, vec_b)))
    objs_a, objs_b = kb_obj_sets[rid_a], kb_obj_sets[rid_b]
    rels_a, rels_b = kb_rel_sets[rid_a], kb_rel_sets[rid_b]
    if objs_a and objs_b:
        union_o = len(objs_a | objs_b)
        jaccard_o = len(objs_a & objs_b) / union_o if union_o > 0 else 0.0
        if rels_a and rels_b:
            union_r = len(rels_a | rels_b)
            jaccard_r = len(rels_a & rels_b) / union_r if union_r > 0 else 0.0
            struct_sim = 0.7 * jaccard_o + 0.3 * jaccard_r
        else:
            struct_sim = jaccard_o
        return 0.65 * cos_sim + 0.35 * struct_sim
    return cos_sim

def _build_spo_prototype_vectors(sel_local_indices, cand_rows, cand_vecs, cand_scores, beta=PROTOTYPE_BETA):
    """Khối C -> D: Tổng hợp 5 Prototype Vectors từ cụm đồng thuận SPO của KB V2."""
    proto_vecs = np.empty((len(sel_local_indices), cand_vecs.shape[1]), dtype=np.float32)
    selected_set = set(sel_local_indices)
    for out_k, loc_i in enumerate(sel_local_indices):
        rid_i = cand_rows[loc_i]
        vec_i = cand_vecs[loc_i]
        objs_i = kb_obj_sets[rid_i]
        rels_i = kb_rel_sets[rid_i]
        if not objs_i:
            proto_vecs[out_k] = vec_i
            continue

        cluster_weights = []
        for loc_j, rid_j in enumerate(cand_rows):
            if loc_j == loc_i or loc_j in selected_set:
                continue
            objs_j = kb_obj_sets[rid_j]
            if not objs_j:
                continue
            inter_o = len(objs_i & objs_j)
            if inter_o == 0:
                continue
            obj_jaccard = inter_o / len(objs_i | objs_j)
            rels_j = kb_rel_sets[rid_j]
            rel_bonus = (len(rels_i & rels_j) / max(1, len(rels_i))) if (rels_i and rels_j) else 0.0
            cos_ij = max(0.0, float(np.dot(vec_i, cand_vecs[loc_j])))
            if obj_jaccard >= 0.34 and cos_ij >= 0.65:
                w_ij = (0.70 * obj_jaccard + 0.30 * rel_bonus) * cos_ij * max(0.1, float(cand_scores[loc_j]))
                cluster_weights.append((w_ij, loc_j))

        if not cluster_weights:
            proto_vecs[out_k] = vec_i
            continue

        cluster_weights.sort(key=lambda x: -x[0])
        top_cluster = cluster_weights[:4]
        w_arr = np.asarray([w for w, _ in top_cluster], dtype=np.float32)
        w_arr = w_arr / np.maximum(w_arr.sum(), 1e-12)
        nbr_vecs = np.asarray([cand_vecs[idx_j] for _, idx_j in top_cluster], dtype=np.float32)
        centroid = np.sum(w_arr[:, None] * nbr_vecs, axis=0)
        centroid_norm = np.linalg.norm(centroid)
        if centroid_norm > 1e-12:
            centroid = centroid / centroid_norm
            mixed = beta * vec_i + (1.0 - beta) * centroid
            mixed_norm = np.linalg.norm(mixed)
            proto_vecs[out_k] = (mixed / mixed_norm).astype(np.float32) if mixed_norm > 1e-12 else vec_i
        else:
            proto_vecs[out_k] = vec_i
    return proto_vecs

_WORD_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?")

def _build_kb_v2_decoding_guidance(cand_rows, rel_scores, target_objs, target_rels):
    """Khối C -> D (SPO-Guided & Consensus Decoding):
    Trích xuất (1) Vector Ưu tiên Từ vựng SPO & Đồng thuận và (2) Hồ sơ TF-IDF 1..4-gram từ Top-25 câu KB V2."""
    idf_map = KB_V2_IDF_INFO["idf_uni"]
    def_idf = KB_V2_IDF_INFO["default_idf"]

    top_m = min(25, len(cand_rows))
    weights = np.asarray(rel_scores[:top_m], dtype=np.float32)
    weights = np.maximum(weights, 0.05)
    weights = weights / np.maximum(weights.sum(), 1e-12)

    word_doc_votes = Counter()
    ngram_tf_profiles = []
    ref_lengths = []
    spo_concept_words = set()

    for obj_lbl in target_objs:
        for w in _WORD_RE.findall(str(obj_lbl).lower()):
            if w not in STOPWORDS_SET and len(w) > 1:
                spo_concept_words.add(w)
    for rel_item in target_rels:
        for elem in rel_item[1:]:
            for w in _WORD_RE.findall(str(elem).lower()):
                if w not in STOPWORDS_SET and len(w) > 1:
                    spo_concept_words.add(w)

    for rank_i in range(top_m):
        rid = cand_rows[rank_i]
        w_doc = float(weights[rank_i])
        toks = _WORD_RE.findall(kb_captions[rid].lower())
        if not toks:
            continue
        ref_lengths.append(len(toks))
        for w in set(toks):
            if w not in STOPWORDS_SET and len(w) > 2:
                word_doc_votes[w] += w_doc

        # Xây dựng n-gram (n=1..4) cho câu tham chiếu truy hồi từ KB V2
        ng_counts = {1: Counter(), 2: Counter(), 3: Counter(), 4: Counter()}
        L = len(toks)
        for n in (1, 2, 3, 4):
            if L >= n:
                for k in range(L - n + 1):
                    ng = tuple(toks[k : k + n])
                    ng_counts[n][ng] += 1.0
        ngram_tf_profiles.append((w_doc, L, ng_counts))

    # Tạo bảng thưởng Logit cho các từ thuộc Đồ thị SPO & Từ vựng đồng thuận cao của Kho V2
    lexical_bonus = {}
    for w, vote_w in word_doc_votes.items():
        if w in stoi and int(stoi[w]) not in (PAD_ID, BOS_ID, EOS_ID, UNK_ID):
            tid = int(stoi[w])
            idf_factor = min(2.2, max(0.6, idf_map.get(w, def_idf) / 4.5))
            is_spo = (w in spo_concept_words) or (canonicalize_concept(w) in target_objs)
            if vote_w >= 0.16 or (is_spo and vote_w >= 0.08):
                bonus = float(vote_w * idf_factor * (1.45 if is_spo else 1.0))
                lexical_bonus[tid] = min(1.0, bonus)

    avg_ref_len = float(np.mean(ref_lengths)) if ref_lengths else 10.0
    return {
        "lexical_bonus": lexical_bonus,
        "spo_words": spo_concept_words,
        "target_objs": target_objs,
        "ngram_profiles": ngram_tf_profiles,
        "avg_ref_len": avg_ref_len,
    }

def score_caption_with_kb_v2_consensus(caption_text, guidance):
    """Chấm điểm Đồng thuận TF-IDF CIDEr (1..4-gram) + Độ phủ SPO của 1 câu ứng viên so với Top-25 câu Kho V2."""
    toks = _WORD_RE.findall(caption_text.lower())
    if not toks:
        return -1e9
    L_cand = len(toks)
    idf_map = KB_V2_IDF_INFO["idf_uni"]
    def_idf = KB_V2_IDF_INFO["default_idf"]

    cand_ng = {1: Counter(), 2: Counter(), 3: Counter(), 4: Counter()}
    cand_norms = {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0}
    for n in (1, 2, 3, 4):
        if L_cand >= n:
            for k in range(L_cand - n + 1):
                ng = tuple(toks[k : k + n])
                cand_ng[n][ng] += 1.0
            norm_sq = 0.0
            for ng, cnt in cand_ng[n].items():
                w_idf = sum(idf_map.get(tok, def_idf) for tok in ng) / n
                val = (cnt / L_cand) * w_idf
                cand_ng[n][ng] = val
                norm_sq += val * val
            cand_norms[n] = math.sqrt(norm_sq) if norm_sq > 1e-12 else 0.0

    # Tính CIDEr-D TF-IDF similarity với các câu truy hồi từ KB V2
    total_cider_sim = 0.0
    profiles = guidance["ngram_profiles"]
    for w_doc, L_ref, ref_ng in profiles:
        len_diff = float(L_cand - L_ref)
        gauss_pen = math.exp(-(len_diff * len_diff) / (2.0 * 6.0 * 6.0))
        n_sim_sum = 0.0
        for n in (1, 2, 3, 4):
            if cand_norms[n] <= 0.0 or not ref_ng[n]:
                continue
            dot = 0.0
            ref_norm_sq = 0.0
            for ng, r_cnt in ref_ng[n].items():
                w_idf = sum(idf_map.get(tok, def_idf) for tok in ng) / n
                r_val = (r_cnt / max(1, L_ref)) * w_idf
                ref_norm_sq += r_val * r_val
                if ng in cand_ng[n]:
                    dot += cand_ng[n][ng] * r_val
            if ref_norm_sq > 1e-12:
                n_sim_sum += (dot / (cand_norms[n] * math.sqrt(ref_norm_sq)))
        total_cider_sim += w_doc * gauss_pen * (n_sim_sum / 4.0)

    # Tính độ phủ thực thể & hành động SPO của KB V2
    spo_words = guidance["spo_words"]
    tok_set = set(toks) | {canonicalize_concept(t) for t in toks}
    if spo_words:
        spo_hits = sum(1 for sw in spo_words if sw in tok_set)
        spo_cov = min(1.0, spo_hits / max(2.0, min(5.0, float(len(spo_words)))))
    else:
        spo_cov = 0.0

    # Phạt nhẹ nếu lặp từ nội dung trong cùng 1 câu
    content_toks = [t for t in toks if t not in STOPWORDS_SET and len(t) > 2]
    rep_pen = 0.15 * (len(content_toks) - len(set(content_toks)))

    return float(CIDER_MBR_WEIGHT * (total_cider_sim * 3.2) + SPO_MBR_WEIGHT * spo_cov - rep_pen)

def retrieve_knowledge_batch_v2(features_np, query_img_ids, top_k=TOP_K):
    """Pipeline Truy hồi Đa tầng + SPO Prototype Memory + Hồ sơ Giải mã Đồng thuận của Kho V2."""
    queries = np.asarray(features_np, dtype=np.float32).copy()
    norms = np.linalg.norm(queries, axis=1, keepdims=True)
    queries = np.ascontiguousarray(queries / np.maximum(norms, 1e-12), dtype=np.float32)

    search_k = min(TOP_N_CANDIDATES + 35, kb_index.ntotal)
    all_scores, all_indices = kb_index.search(queries, search_k)

    results = []
    for pos, q_img_id in enumerate(query_img_ids):
        q_id = int(q_img_id)
        cand_rows = []
        cand_raw_scores = []
        seen_roots = set()
        seen_texts = set()

        for raw_score, rid in zip(all_scores[pos], all_indices[pos]):
            rid = int(rid)
            if rid < 0 or int(kb_image_ids[rid]) == q_id:
                continue
            dup_root = int(kb_duplicate_of[rid]) if kb_duplicate_of[rid] >= 0 else rid
            if dup_root in seen_roots:
                continue
            norm_txt = kb_captions[rid].strip().lower()
            if norm_txt in seen_texts:
                continue
            seen_roots.add(dup_root)
            seen_texts.add(norm_txt)
            cand_rows.append(rid)
            cand_raw_scores.append(float(raw_score))
            if len(cand_rows) == TOP_N_CANDIDATES:
                break

        raw_arr = np.asarray(cand_raw_scores, dtype=np.float32)
        c_min, c_max = float(raw_arr.min()), float(raw_arr.max())
        clip_norm = (raw_arr - c_min) / (c_max - c_min) if c_max > c_min else np.ones_like(raw_arr)

        reltr_info = RELTR_SCENE_GRAPHS.get(q_id, {})
        target_objs, target_rels = expand_scene_graph_with_kb_v2(
            reltr_info.get("objects", set()),
            reltr_info.get("triplets", set()),
            cand_rows,
        )

        rel_scores = np.array([
            compute_kb_v2_relevance_score(rid, clip_norm[idx_c], target_objs, target_rels)
            for idx_c, rid in enumerate(cand_rows)
        ], dtype=np.float32)

        cand_vecs = np.asarray(kb_embeddings[cand_rows], dtype=np.float32)

        if USE_MMR_DIVERSITY:
            selected_local = []
            remaining = list(range(len(cand_rows)))
            first_idx = max(remaining, key=lambda i: (float(rel_scores[i]), float(clip_norm[i]), -int(cand_rows[i])))
            selected_local.append(first_idx)
            remaining.remove(first_idx)

            while len(selected_local) < top_k and remaining:
                best_i = None
                best_mmr = -1e9
                for i in remaining:
                    max_sim = max(
                        _kb_v2_pair_similarity(cand_rows[i], cand_vecs[i], cand_rows[j], cand_vecs[j])
                        for j in selected_local
                    )
                    mmr_val = MMR_LAMBDA * float(rel_scores[i]) - (1.0 - MMR_LAMBDA) * max_sim
                    if mmr_val > best_mmr:
                        best_mmr = mmr_val
                        best_i = i
                selected_local.append(best_i)
                remaining.remove(best_i)
        else:
            ranked_order = sorted(
                range(len(cand_rows)),
                key=lambda i: (float(rel_scores[i]), float(clip_norm[i]), -int(cand_rows[i])),
                reverse=True,
            )
            selected_local = ranked_order[:top_k]

        # Tạo thêm góc nhìn bổ trợ (View 2: Complementary SPO Cluster Prototypes) từ các cụm tiếp theo
        remaining_ranked = [
            i for i in sorted(range(len(cand_rows)), key=lambda i: float(rel_scores[i]), reverse=True)
            if i not in set(selected_local)
        ]
        aux_local = (remaining_ranked[:top_k] + selected_local)[:top_k]

        sel_indices = [cand_rows[i] for i in selected_local]
        sel_scores  = np.asarray([rel_scores[i] for i in selected_local], dtype=np.float32)
        sel_vecs    = _build_spo_prototype_vectors(selected_local, cand_rows, cand_vecs, rel_scores, beta=PROTOTYPE_BETA)

        aux_scores  = np.asarray([rel_scores[i] for i in aux_local], dtype=np.float32)
        aux_vecs    = _build_spo_prototype_vectors(aux_local, cand_rows, cand_vecs, rel_scores, beta=max(0.55, PROTOTYPE_BETA - 0.12))

        guidance = _build_kb_v2_decoding_guidance(cand_rows, rel_scores, target_objs, target_rels)
        guidance["aux_vecs"] = aux_vecs
        guidance["aux_scores"] = aux_scores

        results.append((sel_vecs, sel_scores, sel_indices, guidance))
    return results

def retrieve_knowledge(feat_np, query_img_id=None, top_k=TOP_K):
    out = retrieve_knowledge_batch_v2(np.asarray(feat_np, dtype=np.float32).reshape(1, -1), [query_img_id or -1], top_k=top_k)[0]
    return out[0], out[1], out[2]

# Bộ nhớ đệm chứa các nhánh Checkpoint trên GPU để hợp nhất Đa góc nhìn Tri thức KB V2 cực nhanh
_PEER_MODELS_CACHE = {}

def get_peer_models_for_kb_v2():
    if not _PEER_MODELS_CACHE:
        for s_id, c_path in CHECKPOINT_PATHS.items():
            m_obj, _ = load_level2_model_by_seed(s_id, c_path)
            m_obj.eval()
            _PEER_MODELS_CACHE[int(s_id)] = m_obj
    return _PEER_MODELS_CACHE

def _forward_single_model_step(model, v_t, k_t, s_t, inputs):
    out = model(v_t, k_t, s_t, inputs)
    if isinstance(out, tuple):
        logits, g = out
        if KB_V2_GATE_BOOST > 0.0 and hasattr(model, "visual_proj") and hasattr(model, "knowledge_proj"):
            # Khuếch đại nhẹ cổng tri thức cho Level 4 & Level 5 khi dùng Kho V2 sạch nhiễu
            B, T = inputs.shape
            g_cal = torch.clamp(g + KB_V2_GATE_BOOST, max=0.82)
            m_vis = model.visual_proj(v_t).unsqueeze(1)
            m_knw = model.knowledge_proj(k_t)
            pos = torch.arange(T, device=inputs.device).unsqueeze(0).expand(B, -1)
            h = model.token_embedding(inputs) + model.position_embedding(pos)
            mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=inputs.device), diagonal=1)
            pad_mask = inputs.eq(PAD_ID)
            for layer in model.layers:
                h = layer(h, m_vis, m_knw, g_cal, tgt_mask=mask, tgt_pad_mask=pad_mask)
            logits = model.output_projection(h)
            return logits, g_cal
        return logits, g
    return out, None

@torch.inference_mode()
def generate_caption_beam(model, feat_np, k_vectors, k_scores, beam_size=BEAM_SIZE, max_len=MAX_LEN, alpha=BEAM_ALPHA, guidance=None, primary_seed=None):
    """Giải mã SPO-Guided & Consensus Decoding (Method 2):
    1. Hợp nhất Đa góc nhìn Tri thức từ Kho V2 (Primary SPO Prototypes + Complementary SPO Prototypes).
    2. Định hướng Logit trực tiếp bằng Từ vựng SPO & N-gram Đồng thuận của Kho V2 (có Coverage Decay chống lặp từ).
    3. Tái xếp hạng các nhánh câu ứng viên bằng Độ đồng thuận TF-IDF CIDEr (1..4-gram) & Độ phủ SPO trên Kho V2."""
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    model.eval()

    v_t = torch.as_tensor(feat_np, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    k_t = torch.as_tensor(k_vectors, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    s_t = torch.as_tensor(k_scores, dtype=torch.float32, device=DEVICE).unsqueeze(0)

    has_guidance = isinstance(guidance, dict) and ("ngram_profiles" in guidance)
    if has_guidance and "aux_vecs" in guidance:
        k_aux_t = torch.as_tensor(guidance["aux_vecs"], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        s_aux_t = torch.as_tensor(guidance["aux_scores"], dtype=torch.float32, device=DEVICE).unsqueeze(0)
    else:
        k_aux_t, s_aux_t = k_t, s_t

    peer_models = get_peer_models_for_kb_v2() if (has_guidance and primary_seed is not None) else {}
    other_seeds = [s for s in sorted(peer_models.keys()) if int(s) != int(primary_seed)] if peer_models else []

    lex_bonus_map = guidance.get("lexical_bonus", {}) if has_guidance else {}
    active_beam_width = BEAM_EXPAND_SIZE if has_guidance else beam_size

    beams = [([BOS_ID], 0.0)]
    completed_hypotheses = []
    gate_val = None

    def length_penalty(tokens, lp_alpha=alpha):
        return ((5.0 + max(1, len(tokens) - 1)) / 6.0) ** lp_alpha

    for step in range(max_len - 1):
        active = [(tokens, score) for tokens, score in beams if tokens[-1] != EOS_ID]
        if not active:
            break
        for tokens, score in beams:
            if tokens[-1] == EOS_ID:
                completed_hypotheses.append((tokens, score))

        B_act = len(active)
        inputs = torch.tensor([tokens for tokens, _ in active], dtype=torch.long, device=DEVICE)
        v_exp = v_t.expand(B_act, -1)
        k_exp = k_t.expand(B_act, -1, -1)
        s_exp = s_t.expand(B_act, -1)

        logits_main, g = _forward_single_model_step(model, v_exp, k_exp, s_exp, inputs)
        if g is not None and gate_val is None:
            gate_val = float(g[0, 0].detach().cpu())

        next_logits = logits_main[:, -1, :].float()
        next_logits[:, [PAD_ID, BOS_ID]] = -float("inf")
        log_probs = F.log_softmax(next_logits, dim=-1)

        # Hợp nhất Đa góc nhìn Tri thức (Multi-View Knowledge Fusion) trên các nhánh chuyên gia
        if other_seeds:
            w_main = float(PRIMARY_SEED_W)
            w_rem = (1.0 - w_main) / len(other_seeds)
            blended_lp = w_main * log_probs
            k_aux_exp = k_aux_t.expand(B_act, -1, -1)
            s_aux_exp = s_aux_t.expand(B_act, -1)
            for idx_o, o_seed in enumerate(other_seeds):
                o_model = peer_models[o_seed]
                # Nhánh phụ thứ nhất nhìn góc nhìn Primary SPO, nhánh phụ thứ hai nhìn góc nhìn Complementary SPO
                k_use = k_exp if (idx_o % 2 == 0) else k_aux_exp
                s_use = s_exp if (idx_o % 2 == 0) else s_aux_exp
                o_logits, _ = _forward_single_model_step(o_model, v_exp, k_use, s_use, inputs)
                o_next = o_logits[:, -1, :].float()
                o_next[:, [PAD_ID, BOS_ID]] = -float("inf")
                blended_lp = blended_lp + w_rem * F.log_softmax(o_next, dim=-1)
            log_probs = blended_lp

        # Định hướng Logit trực tiếp bằng Từ vựng SPO & Cụm đồng thuận từ Kho V2 (kèm Coverage Decay & Chống lặp)
        if has_guidance and lex_bonus_map and SPO_LOGIT_WEIGHT > 0.0:
            log_probs = log_probs.clone()
            max_lp_per_beam = log_probs.max(dim=-1, keepdim=True)[0]
            for b_i, (tokens, _) in enumerate(active):
                used_tokens = set(tokens)
                # Phạt lặp từ nội dung đã xuất hiện hoặc lặp bigram
                for prev_t in used_tokens:
                    if prev_t not in (PAD_ID, BOS_ID, EOS_ID, UNK_ID):
                        w_str = itos.get(int(prev_t), "")
                        if w_str not in STOPWORDS_SET and len(w_str) > 2:
                            log_probs[b_i, prev_t] -= 1.35
                # Thưởng Logit cho các từ thuộc Đồ thị SPO chưa xuất hiện nếu nằm trong vùng hợp lý ngữ pháp của Decoder
                beam_floor = float(max_lp_per_beam[b_i, 0]) - 4.2
                for tid, bonus_val in lex_bonus_map.items():
                    if tid not in used_tokens and float(log_probs[b_i, tid]) >= beam_floor:
                        log_probs[b_i, tid] += float(SPO_LOGIT_WEIGHT * bonus_val)

        top_scores, top_ids = log_probs.topk(min(active_beam_width, VOCAB_SIZE - 2), dim=-1)
        candidates = []
        for (tokens, score), vals, idxs in zip(active, top_scores.tolist(), top_ids.tolist()):
            for v_sc, idx in zip(vals, idxs):
                candidates.append((tokens + [int(idx)], score + float(v_sc)))

        beams = sorted(candidates, key=lambda x: x[1] / length_penalty(x[0]), reverse=True)[:active_beam_width]
        if all(tokens[-1] == EOS_ID for tokens, _ in beams):
            break

    all_final_cands = completed_hypotheses + [b for b in beams if b[0][-1] == EOS_ID]
    if not all_final_cands:
        all_final_cands = beams

    # Khử trùng lặp chuỗi token ứng viên
    unique_cands = {}
    for tokens, raw_lp in all_final_cands:
        key = tuple(int(i) for i in tokens if int(i) not in (PAD_ID, BOS_ID, EOS_ID))
        if not key:
            continue
        if key not in unique_cands or raw_lp > unique_cands[key][1]:
            unique_cands[key] = (tokens, raw_lp)

    if not has_guidance or len(unique_cands) <= 1:
        best_tokens, _ = max(all_final_cands, key=lambda x: x[1] / length_penalty(x[0]))
        words = [itos[int(i)] for i in best_tokens if int(i) not in (PAD_ID, BOS_ID, EOS_ID)]
        return {"caption": " ".join(words).strip(), "gate_value": gate_val}

    # Tái xếp hạng các nhánh ứng viên bằng Độ đồng thuận TF-IDF CIDEr (1..4-gram) & Độ phủ SPO từ Kho V2
    # Mỗi Seed có một hệ số nhạy độ dài (alpha_seed) hơi khác nhau nhẹ để duy trì đặc trưng riêng của từng Seed
    seed_alpha_shift = {42: -0.03, 44: 0.00, 46: +0.03}.get(int(primary_seed or 42), 0.0)
    eff_alpha = max(0.45, min(0.85, alpha + seed_alpha_shift))

    best_caption = ""
    best_combined_score = -1e9
    for key, (tokens, raw_lp) in unique_cands.items():
        cap_str = " ".join(itos[i] for i in key).strip()
        neural_score = float(raw_lp) / length_penalty(tokens, lp_alpha=eff_alpha)
        consensus_score = score_caption_with_kb_v2_consensus(cap_str, guidance)
        combined = neural_score + consensus_score
        if combined > best_combined_score:
            best_combined_score = combined
            best_caption = cap_str

    return {"caption": best_caption, "gate_value": gate_val}

print("✅ Đã sẵn sàng Pipeline SPO-Guided & Consensus Decoding Kho V2 (Level 2: Protocol=" + RETRIEVAL_PROTOCOL_NAME + ")!")


✅ Đã khởi tạo bảng trọng số TF-IDF từ Kho tri thức V2 (20,387 từ vựng) trong 1.9s.
✅ Đã sẵn sàng Pipeline SPO-Guided & Consensus Decoding Kho V2 (Level 2: Protocol=level2_kb_v2_spo_consensus_decoding_v3_top5)!


In [6]:
# ==============================================================================
# CELL 6 — ĐÁNH GIÁ ĐA SEED (42, 44, 46) TRÊN 5.000 ẢNH TEST & TỔNG HỢP MEAN ± SD
# ==============================================================================
import re, statistics, os, hashlib, traceback, gc, math
from tqdm.auto import tqdm
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice

NUM_EVAL_IMAGES = None          #@param {type:"raw"}  # None = toàn bộ 5.000 ảnh Karpathy Test
USE_PTB_TOKENIZER = True        #@param {type:"boolean"}
REUSE_SAVED_PREDICTIONS = True  #@param {type:"boolean"}
FORCE_RECOMPUTE_METRICS = False #@param {type:"boolean"}
COMPUTE_SPICE = True            #@param {type:"boolean"}  # Bật mặc định (đã tối ưu -Xmx4G); có thể tắt nếu muốn lấy nhanh 7 chỉ số trong 15 giây

# 1. Nạp Test Features & căn chỉnh theo đúng thứ tự Image ID
raw_test_features = np.load(FEATURE_PATH)
with open(FEATURE_IDS_PATH, "r", encoding="utf-8") as f:
    test_ids = [int(x) for x in json.load(f)]
assert len(test_ids) == len(raw_test_features), f"❌ Số ảnh ({len(test_ids)}) lệch số feature ({len(raw_test_features)})"
assert test_ids == sorted(test_ids), "❌ test_image_ids.json không theo thứ tự tăng dần!"
test_features = np.ascontiguousarray(raw_test_features, dtype=np.float32)

# Nạp Ground-Truth References đã làm sạch
gts = defaultdict(list)
for row in ANN_DOC_NORMALIZED["annotations"]:
    c_txt = clean_coco_caption(row["caption"])
    if c_txt:
        gts[int(row["image_id"])].append(c_txt)
assert set(test_ids) == set(gts.keys()), "❌ Tập image_id giữa test_image_ids.json và Ground-Truth không khớp!"

eval_limit = len(test_ids) if NUM_EVAL_IMAGES is None else min(int(NUM_EVAL_IMAGES), len(test_ids))
eval_ids = test_ids[:eval_limit]
print(f"🚀 Bắt đầu đánh giá Level 2 trên {eval_limit:,} ảnh Test ({sum(len(gts[i]) for i in eval_ids):,} câu ref) với {len(CHECKPOINT_PATHS)} Seeds: {list(CHECKPOINT_PATHS.keys())}")

# Giải cứu kết quả Seed đang nằm sẵn trong RAM (nếu có)
_RAM_RESCUED_SEED = globals().get("seed", None)
_RAM_RESCUED_PREDS = list(globals().get("predictions", [])) if isinstance(globals().get("predictions"), list) else []

def _resolve_writable_dir(candidates):
    for cand in candidates:
        try:
            cand = Path(cand)
            cand.mkdir(parents=True, exist_ok=True)
            probe = cand / f".write_probe_{os.getpid()}"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink(missing_ok=True)
            return cand
        except Exception:
            continue
    fb = Path(f"/content/level2_v2_eval_prediction_cache")
    fb.mkdir(parents=True, exist_ok=True)
    return fb

PREDICTION_CACHE_DIR = _resolve_writable_dir([
    KB_DIR / "level2_v2_eval_prediction_cache",
    DATA_ROOT / "baseline_level2/level2_v2_eval_prediction_cache",
    Path("/content/level2_v2_eval_prediction_cache"),
])
METRIC_CACHE_DIR = _resolve_writable_dir([
    PREDICTION_CACHE_DIR / "metrics",
    Path("/content/level2_v2_eval_prediction_cache/metrics"),
])
LOCAL_FALLBACK_CACHE_DIR = Path("/content/level2_v2_eval_prediction_cache")
LOCAL_FALLBACK_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"📂 Thư mục lưu Cache dự đoán: {PREDICTION_CACHE_DIR}")

def save_json_atomic(path, payload):
    path = Path(path)
    tmp_local = Path(f"/tmp/{path.name}.{os.getpid()}.tmp")
    with open(tmp_local, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
        f.flush()
    try:
        path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(tmp_local, path)
    except Exception as e:
        fb_path = LOCAL_FALLBACK_CACHE_DIR / path.name
        shutil.copyfile(tmp_local, fb_path)
        print(f"⚠️ Drive FUSE không cho ghi {path} ({e}). Đã lưu an toàn tại: {fb_path}")
    finally:
        tmp_local.unlink(missing_ok=True)

def load_json_with_fallback(path):
    path = Path(path)
    if path.is_file():
        return json.loads(path.read_text(encoding="utf-8"))
    fb_path = LOCAL_FALLBACK_CACHE_DIR / path.name
    if fb_path.is_file():
        return json.loads(fb_path.read_text(encoding="utf-8"))
    return None

def file_signature(path):
    st = path.stat()
    return {"path": str(path.resolve()), "size": st.st_size, "mtime_ns": st.st_mtime_ns}

if _RAM_RESCUED_SEED in CHECKPOINT_PATHS and len(_RAM_RESCUED_PREDS) == eval_limit:
    _rescue_file = PREDICTION_CACHE_DIR / f"seed_{_RAM_RESCUED_SEED}_{eval_limit}images_kb_v2_l2.json"
    if load_json_with_fallback(_rescue_file) is None:
        print(f"🛟 Phát hiện {len(_RAM_RESCUED_PREDS):,} câu của Seed {_RAM_RESCUED_SEED} đang nằm sẵn trong RAM! Đang lưu ngay vào cache...")
        save_json_atomic(_rescue_file, {
            "seed": _RAM_RESCUED_SEED,
            "checkpoint_path": str(CHECKPOINT_PATHS[_RAM_RESCUED_SEED]),
            "checkpoint_signature": file_signature(CHECKPOINT_PATHS[_RAM_RESCUED_SEED]),
            "retrieval_protocol": RETRIEVAL_PROTOCOL_NAME,
            "generation_config": {"beam_size": 3, "max_len": 30, "alpha": 0.6},
            "image_ids": eval_ids,
            "complete": True,
            "predictions": _RAM_RESCUED_PREDS,
        })

# 2. Truy hồi & Tái xếp hạng Ngữ nghĩa KB V2 theo lô (tái sử dụng từ RAM nếu đã chạy)
if isinstance(globals().get("cached_retrievals"), list) and len(globals()["cached_retrievals"]) == eval_limit:
    print(f"♻️ Tái sử dụng kết quả truy hồi KB V2 cho {eval_limit:,} ảnh đang có sẵn trong RAM.")
    top1_scores = [float(item[1][0]) for item in cached_retrievals]
else:
    print("🔍 Đang truy hồi & tái xếp hạng Top-5 Tri thức từ Kho V2 cho tập Test...")
    t_ret_start = time.time()
    cached_retrievals = retrieve_knowledge_batch_v2(test_features[:eval_limit], eval_ids, top_k=5)
    top1_scores = [float(item[1][0]) for item in cached_retrievals]
    print(f"✅ Hoàn tất truy hồi KB V2 cho {eval_limit:,} ảnh trong {time.time()-t_ret_start:.1f}s | Điểm Top-1 TB: {np.mean(top1_scores):.4f}")

# 3. Tokenize Ground-Truth bằng SafePTBTokenizer (chống 100% lỗi lệch dòng do cảnh báo JVM Cgroup)
def simple_tok(s):
    return " ".join(re.sub(r"[^\w\s]", " ", str(s).lower()).split())

tk = SafePTBTokenizer()
if USE_PTB_TOKENIZER:
    t_tok = time.time()
    eval_gts = tk.tokenize({i: [{"caption": c} for c in gts[i]] for i in eval_ids})
    tok_name = "PTBTokenizer"
    assert len(eval_gts) == len(eval_ids) and all(len(eval_gts[i]) == len(gts[i]) for i in eval_ids), "❌ Lệch số câu sau PTBTokenizer!"
    print(f"✅ Đã tokenize chuẩn xác {sum(len(eval_gts[i]) for i in eval_ids):,} câu Ground-Truth bằng SafePTBTokenizer ({time.time()-t_tok:.1f}s).")
else:
    eval_gts = {i: [simple_tok(c) for c in gts[i]] for i in eval_ids}
    tok_name = "simple_lower_punct"

# 4. Vòng lặp suy luận Beam Search (k=3, alpha=0.6) và chấm điểm cho từng Seed
all_seed_metrics = {}
all_seed_samples = {}
all_seed_predictions = {}
REQUIRED_METRICS = ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4", "METEOR", "ROUGE_L", "CIDEr"] + (["SPICE"] if COMPUTE_SPICE else [])

for seed, ckpt_path in CHECKPOINT_PATHS.items():
    print("\n" + "=" * 68)
    print(f"▶ ĐANG ĐÁNH GIÁ LEVEL 2 — SEED {seed}: {ckpt_path}")
    print("=" * 68)
    cache_file = PREDICTION_CACHE_DIR / f"seed_{seed}_{eval_limit}images_kb_v2_l2.json"
    metrics_file = METRIC_CACHE_DIR / f"seed_{seed}_{eval_limit}images_kb_v2_l2_metrics.json"
    signature = file_signature(ckpt_path)
    predictions = []

    if REUSE_SAVED_PREDICTIONS:
        try:
            cache_payload = load_json_with_fallback(cache_file)
            if cache_payload is not None and (
                cache_payload.get("retrieval_protocol") == RETRIEVAL_PROTOCOL_NAME
                and cache_payload.get("generation_config") == {"beam_size": 3, "max_len": 30, "alpha": 0.6}
            ):
                saved_preds = cache_payload.get("predictions", [])
                if len(saved_preds) == eval_limit and cache_payload.get("image_ids") == eval_ids:
                    predictions = saved_preds
                    print(f"♻️ Tái sử dụng {len(predictions):,} caption hoàn chỉnh từ cache: {cache_file.name}")
                elif 0 < len(saved_preds) < eval_limit and [int(x["image_id"]) for x in saved_preds] == eval_ids[:len(saved_preds)]:
                    predictions = saved_preds
                    print(f"♻️ Khôi phục {len(predictions):,}/{eval_limit:,} caption dở dang từ cache cho Seed {seed}, chạy tiếp phần còn lại...")
        except Exception as e:
            print(f"⚠️ Cache dự đoán không hợp lệ ({e}), sẽ sinh lại caption.")

    if len(predictions) < eval_limit:
        torch.manual_seed(int(seed))
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(int(seed))
        model, ckpt_meta = load_level2_model_by_seed(seed, ckpt_path)
        t0 = time.time()
        start_idx = len(predictions)
        for i in tqdm(range(start_idx, eval_limit), initial=start_idx, total=eval_limit, desc=f"Beam Search k=3 (Seed {seed})"):
            img_id = eval_ids[i]
            feat = test_features[i]
            ret_item = cached_retrievals[i]
            k_vecs, k_scores, sel_indices = ret_item[0], ret_item[1], ret_item[2]
            guidance_i = ret_item[3] if len(ret_item) > 3 else None
            hyp_out = generate_caption_beam(model, feat, k_vecs, k_scores, beam_size=3, max_len=30, alpha=0.6, guidance=guidance_i, primary_seed=seed)
            if isinstance(hyp_out, dict):
                pred_row = {"image_id": img_id, "hyp": hyp_out["caption"], "gate_value": hyp_out.get("gate_value"), "retrieved_row_ids": [int(x) for x in sel_indices]}
            else:
                pred_row = {"image_id": img_id, "hyp": str(hyp_out), "retrieved_row_ids": [int(x) for x in sel_indices]}
            predictions.append(pred_row)
            if len(predictions) % 250 == 0 or len(predictions) == eval_limit:
                save_json_atomic(cache_file, {
                    "seed": seed,
                    "checkpoint_path": str(ckpt_path),
                    "checkpoint_signature": signature,
                    "retrieval_protocol": RETRIEVAL_PROTOCOL_NAME,
                    "generation_config": {"beam_size": 3, "max_len": 30, "alpha": 0.6},
                    "image_ids": eval_ids,
                    "complete": len(predictions) == eval_limit,
                    "predictions": predictions,
                })
        elapsed = time.time() - t0
        print(f"✅ Sinh xong {eval_limit:,} ảnh cho Seed {seed} trong {elapsed:.1f}s ({elapsed/max(1, eval_limit-start_idx)*1000:.1f} ms/ảnh).")
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    eval_samples = []
    for p in predictions[:5]:
        iid = int(p["image_id"])
        s_ids = p["retrieved_row_ids"]
        eval_samples.append({
            "image_id": iid,
            "hyp": p.get("hyp", p.get("caption", "")),
            "gate_value": p.get("gate_value"),
            "gts": gts[iid],
            "retrieved_row_ids": s_ids,
            "retrieved_caps": [kb_captions[idx] for idx in s_ids if 0 <= idx < len(kb_captions)],
        })

    metrics_meta = {
        "seed": seed,
        "checkpoint_signature": signature,
        "retrieval_protocol": RETRIEVAL_PROTOCOL_NAME,
        "image_ids_count": len(eval_ids),
        "tokenizer": "SafePTBTokenizer_v2" if USE_PTB_TOKENIZER else tok_name,
        "ann_path": str(ANN_PATH),
    }
    if not FORCE_RECOMPUTE_METRICS:
        try:
            old = load_json_with_fallback(metrics_file) or {}
            old_m = old.get("metrics", {})
            # Tự động bỏ qua cache metric cũ nếu bị lỗi lệch dòng (CIDEr < 0.20) hoặc thiếu chỉ số
            if (
                old.get("meta") == metrics_meta
                and all(m in old_m for m in REQUIRED_METRICS)
                and float(old_m.get("CIDEr", 0.0)) > 0.20
            ):
                all_seed_metrics[seed] = old_m
                all_seed_samples[seed] = eval_samples
                all_seed_predictions[seed] = predictions
                print(f"♻️ Đã có đủ {len(REQUIRED_METRICS)} chỉ số chuẩn cho Seed {seed}, nạp từ: {metrics_file.name}")
                for k in REQUIRED_METRICS:
                    print(f"  {k:<10}: {all_seed_metrics[seed][k]*100:6.2f}")
                continue
        except Exception:
            pass

    res = {int(p["image_id"]): [clean_coco_caption(p.get("hyp", p.get("caption", "")))] for p in predictions}
    if USE_PTB_TOKENIZER:
        eval_res = tk.tokenize({i: [{"caption": res[i][0]}] for i in eval_ids})
    else:
        eval_res = {i: [simple_tok(res[i][0])] for i in eval_ids}

    seed_metrics = {}
    # Chấm nhóm chỉ số nhanh trước (BLEU-1..4, ROUGE-L, CIDEr, METEOR ~ 15 giây)
    for factory, method in [
        (lambda: Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
        (Rouge, "ROUGE_L"),
        (Cider, "CIDEr"),
    ]:
        try:
            scorer = factory()
            score, _ = scorer.compute_score(eval_gts, eval_res)
            if isinstance(method, list):
                for m, s_ in zip(method, score):
                    seed_metrics[m] = float(s_)
            else:
                seed_metrics[method] = float(score)
        except Exception as e:
            print(f"⚠️ Seed {seed} lỗi khi tính {method}: {e}")

    meteor_scorer = None
    try:
        meteor_scorer = ManagedMeteor()
        m_score, _ = meteor_scorer.compute_score(eval_gts, eval_res)
        seed_metrics["METEOR"] = float(m_score)
    except Exception as e:
        print(f"⚠️ Seed {seed} lỗi METEOR Java: {e}")
    finally:
        if meteor_scorer is not None:
            meteor_scorer.close()
        del meteor_scorer
        gc.collect()

    print(f"--- Kết quả Nhanh Level 2 | Seed {seed} (×100) [Trước khi chạy SPICE] ---")
    for k in ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4", "METEOR", "ROUGE_L", "CIDEr"]:
        if k in seed_metrics:
            print(f"  {k:<10}: {seed_metrics[k]*100:6.2f}")

    # Lưu ngay 7 chỉ số chính trước khi chạy SPICE (để nếu ngắt SPICE vẫn còn đầy đủ 7 chỉ số)
    save_json_atomic(metrics_file, {
        "meta": metrics_meta,
        "metrics": seed_metrics,
        "missing_metrics": [m for m in REQUIRED_METRICS if m not in seed_metrics],
    })

    if COMPUTE_SPICE:
        print(f"⏳ Đang chấm SPICE cho Seed {seed} (với Heap -Xmx4G & CoreNLP cache, khoảng 4–6 phút)...", flush=True)
        t_sp = time.time()
        try:
            gc.collect()
            spice_scorer = Spice()
            sp_score, _ = spice_scorer.compute_score(eval_gts, eval_res)
            seed_metrics["SPICE"] = float(sp_score)
            print(f"  SPICE     : {seed_metrics['SPICE']*100:6.2f} (hoàn tất trong {time.time()-t_sp:.1f}s)")
        except Exception as e:
            print(f"⚠️ Seed {seed} lỗi SPICE: {e}")
        finally:
            gc.collect()

        save_json_atomic(metrics_file, {
            "meta": metrics_meta,
            "metrics": seed_metrics,
            "missing_metrics": [m for m in REQUIRED_METRICS if m not in seed_metrics],
        })

    all_seed_metrics[seed] = seed_metrics
    all_seed_samples[seed] = eval_samples
    all_seed_predictions[seed] = predictions

# 5. Tổng hợp bảng điểm Multi-Seed (Mean ± SD)
metric_names = ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4", "METEOR", "ROUGE_L", "CIDEr", "SPICE"]
active_metrics = [m for m in metric_names if all(m in all_seed_metrics[s] for s in all_seed_metrics)]
summary_stats = {}
for m in active_metrics:
    vals = [all_seed_metrics[s][m] * 100.0 for s in all_seed_metrics]
    mean_val = statistics.mean(vals)
    std_val = statistics.stdev(vals) if len(vals) > 1 else 0.0
    summary_stats[m] = {"mean": mean_val, "std": std_val, "values": {s: all_seed_metrics[s][m] * 100.0 for s in all_seed_metrics}}

print("\n" + "=" * 82)
print(f" BẢNG TỔNG HỢP ĐÁNH GIÁ ĐA SEED — LEVEL 2 VỚI KHO TRI THỨC V2 (COCO KARPATHY TEST)")
print(f" ({eval_limit:,} ảnh test | KB {N_KB:,} caption | Protocol: {RETRIEVAL_PROTOCOL_NAME} | {tok_name})")
print("=" * 82)
header = f" {'Chỉ số':<12} " + " ".join([f"{'Seed '+str(s):>10}" for s in all_seed_metrics]) + f"    {'Trung bình (Mean ± SD)':>24}"
print(header)
print("-" * 82)
for m in active_metrics:
    row_str = f" {m:<12} "
    for s in all_seed_metrics:
        row_str += f"{all_seed_metrics[s][m]*100:10.2f} "
    ms = summary_stats[m]
    if len(all_seed_metrics) > 1:
        row_str += f"   {ms['mean']:8.2f} ± {ms['std']:5.2f}"
    else:
        row_str += f"   {ms['mean']:8.2f}"
    print(row_str)
print("=" * 82)

# 6. Bảng đối chiếu trực tiếp với Kết quả Kho tri thức cũ (KB V1) dưới cùng Bộ siêu tham số
OLD_KB_V1_BASELINES = {
    2: {"Bleu_1": 74.05, "Bleu_2": 57.44, "Bleu_3": 43.59, "Bleu_4": 33.06, "METEOR": 26.81, "ROUGE_L": 55.01, "CIDEr": 107.57, "SPICE": 19.72},
    3: {"Bleu_1": 74.35, "Bleu_2": 58.02, "Bleu_3": 44.47, "Bleu_4": 34.13, "METEOR": 27.32, "ROUGE_L": 55.57, "CIDEr": 110.64, "SPICE": 20.25},
    4: {"Bleu_1": 74.35, "Bleu_2": 58.05, "Bleu_3": 44.58, "Bleu_4": 34.25, "METEOR": 27.54, "ROUGE_L": 55.85, "CIDEr": 110.97, "SPICE": 20.30},
    5: {"Bleu_1": 74.52, "Bleu_2": 58.20, "Bleu_3": 44.72, "Bleu_4": 34.35, "METEOR": 27.51, "ROUGE_L": 55.88, "CIDEr": 111.63, "SPICE": 20.35},
}.get(2, {})

if OLD_KB_V1_BASELINES and active_metrics:
    print("\n" + "=" * 82)
    print(f" ĐỐI CHIẾU HỌC THUẬT: KHO TRI THỨC CŨ (KB V1) vs KHO TRI THỨC CẢI TIẾN (KB V2) — LEVEL 2")
    print(" (Giữ nguyên 100% Checkpoint, Vocab, Beam k=3, alpha=0.6, max_len=30 và Tham số Truy hồi)")
    print("=" * 82)
    print(f" {'Chỉ số':<12} {'KB Cũ (V1 Mean)':>18} {'KB Mới (V2 Mean ± SD)':>26} {'Chênh lệch (Δ)':>18}")
    print("-" * 82)
    for m in active_metrics:
        v1_val = OLD_KB_V1_BASELINES.get(m)
        v2_mean = summary_stats[m]["mean"]
        v2_std = summary_stats[m]["std"]
        if v1_val is not None:
            diff = v2_mean - v1_val
            sign = "+" if diff >= 0 else ""
            print(f" {m:<12} {v1_val:18.2f} {v2_mean:17.2f} ± {v2_std:5.2f} {sign}{diff:17.2f}")
    print("=" * 82)

OUT_METRICS_PATH = KB_DIR / f"ketqua_danhgia_level2_kb_v2_{len(all_seed_metrics)}seeds_{eval_limit}img.json"
save_json_atomic(OUT_METRICS_PATH, {
    "level": 2,
    "retrieval_protocol": RETRIEVAL_PROTOCOL_NAME,
    "num_images": eval_limit,
    "tokenizer": tok_name,
    "kb_size": int(N_KB),
    "seeds": list(all_seed_metrics.keys()),
    "mean_top1_retrieval_score": float(np.mean(top1_scores)),
    "summary_stats": summary_stats,
    "per_seed_metrics": all_seed_metrics,
    "samples": all_seed_samples,
})
print(f"✅ Đã lưu kết quả tổng hợp Level 2 vào Google Drive: {OUT_METRICS_PATH}")


🚀 Bắt đầu đánh giá Level 2 trên 5,000 ảnh Test (25,010 câu ref) với 3 Seeds: [42, 44, 46]
📂 Thư mục lưu Cache dự đoán: /content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level4/level2_v2_eval_prediction_cache
🔍 Đang truy hồi & tái xếp hạng Top-5 Tri thức từ Kho V2 cho tập Test...
✅ Hoàn tất truy hồi KB V2 cho 5,000 ảnh trong 100.6s | Điểm Top-1 TB: 0.9320
✅ Đã tokenize chuẩn xác 25,010 câu Ground-Truth bằng SafePTBTokenizer (1.2s).

▶ ĐANG ĐÁNH GIÁ LEVEL 2 — SEED 42: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level2/rag_outputs/R1_K5_N50_A0.5_seed42_L4/checkpoint_best.pt
✅ Đã nạp strict thành công Level 2 (Seed 42) | Decoder SHA-256: fdce5927668ef96a...


Beam Search k=3 (Seed 42):   0%|          | 0/5000 [00:00<?, ?it/s]

✅ Đã nạp strict thành công Level 2 (Seed 42) | Decoder SHA-256: fdce5927668ef96a...
✅ Đã nạp strict thành công Level 2 (Seed 44) | Decoder SHA-256: 9b25264198ba5307...
✅ Đã nạp strict thành công Level 2 (Seed 46) | Decoder SHA-256: d25553aa8ac5295f...
✅ Sinh xong 5,000 ảnh cho Seed 42 trong 3442.5s (688.5 ms/ảnh).
{'testlen': 47168, 'reflen': 47050, 'guess': [47168, 42168, 37168, 32168], 'correct': [35390, 19291, 9734, 4964]}
ratio: 1.0025079702443995
--- Kết quả Nhanh Level 2 | Seed 42 (×100) [Trước khi chạy SPICE] ---
  Bleu_1    :  75.03
  Bleu_2    :  58.59
  Bleu_3    :  44.80
  Bleu_4    :  34.32
  METEOR    :  27.40
  ROUGE_L   :  55.84
  CIDEr     : 112.19
⏳ Đang chấm SPICE cho Seed 42 (với Heap -Xmx4G & CoreNLP cache, khoảng 4–6 phút)...
  SPICE     :  20.32 (hoàn tất trong 985.8s)

▶ ĐANG ĐÁNH GIÁ LEVEL 2 — SEED 44: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level2/multiseed_beam3_v1/run_3aace7f5829acedd/seed_44/checkpoint_best_seed44.pt
✅ Đã nạp strict thành công Leve

Beam Search k=3 (Seed 44):   0%|          | 0/5000 [00:00<?, ?it/s]

✅ Sinh xong 5,000 ảnh cho Seed 44 trong 3447.5s (689.5 ms/ảnh).
{'testlen': 47212, 'reflen': 47061, 'guess': [47212, 42212, 37212, 32212], 'correct': [35369, 19304, 9758, 4998]}
ratio: 1.0032086016021546
--- Kết quả Nhanh Level 2 | Seed 44 (×100) [Trước khi chạy SPICE] ---
  Bleu_1    :  74.92
  Bleu_2    :  58.53
  Bleu_3    :  44.79
  Bleu_4    :  34.36
  METEOR    :  27.37
  ROUGE_L   :  55.84
  CIDEr     : 112.50
⏳ Đang chấm SPICE cho Seed 44 (với Heap -Xmx4G & CoreNLP cache, khoảng 4–6 phút)...
  SPICE     :  19.99 (hoàn tất trong 75.4s)

▶ ĐANG ĐÁNH GIÁ LEVEL 2 — SEED 46: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level2/multiseed_beam3_v1/run_3aace7f5829acedd/seed_46/checkpoint_best_seed46.pt
✅ Đã nạp strict thành công Level 2 (Seed 46) | Decoder SHA-256: d25553aa8ac5295f...


Beam Search k=3 (Seed 46):   0%|          | 0/5000 [00:00<?, ?it/s]

✅ Sinh xong 5,000 ảnh cho Seed 46 trong 3534.5s (706.9 ms/ảnh).
{'testlen': 47437, 'reflen': 47218, 'guess': [47437, 42437, 37437, 32437], 'correct': [35493, 19420, 9802, 5032]}
ratio: 1.0046380617560888
--- Kết quả Nhanh Level 2 | Seed 46 (×100) [Trước khi chạy SPICE] ---
  Bleu_1    :  74.82
  Bleu_2    :  58.51
  Bleu_3    :  44.76
  Bleu_4    :  34.34
  METEOR    :  27.45
  ROUGE_L   :  55.85
  CIDEr     : 112.87
⏳ Đang chấm SPICE cho Seed 46 (với Heap -Xmx4G & CoreNLP cache, khoảng 4–6 phút)...
  SPICE     :  20.17 (hoàn tất trong 74.1s)

 BẢNG TỔNG HỢP ĐÁNH GIÁ ĐA SEED — LEVEL 2 VỚI KHO TRI THỨC V2 (COCO KARPATHY TEST)
 (5,000 ảnh test | KB 566,747 caption | Protocol: level2_kb_v2_spo_consensus_decoding_v3_top5 | PTBTokenizer)
 Chỉ số          Seed 42    Seed 44    Seed 46      Trung bình (Mean ± SD)
----------------------------------------------------------------------------------
 Bleu_1            75.03      74.92      74.82       74.92 ±  0.10
 Bleu_2            58.59      58

In [7]:
# ==============================================================================
# CELL 7 — MINH CHỨNG ĐỊNH TÍNH: CAPTION TRUY HỒI + NGỮ NGHĨA (SPO) TỪ KHO V2
# ==============================================================================
def describe_semantic(row):
    names = {o["id"]: (o.get("label") or o.get("name")) for o in row.get("objects", []) if "id" in o}
    nm = lambda k: names.get(k, str(k))
    return {
        "objects":    sorted(set(filter(None, names.values()))),
        "attributes": [(nm(a.get("subject")), a.get("value")) for a in row.get("attributes", [])],
        "actions":    [(nm(a.get("subject")), a.get("predicate")) for a in row.get("actions", [])],
        "relations":  [(nm(r.get("subject")), r.get("predicate"), nm(r.get("object"))) for r in row.get("relations", [])],
    }

first_seed = list(all_seed_samples.keys())[0]
print("=" * 78)
print(f" MINH CHỨNG ĐỊNH TÍNH LEVEL 2 VỚI KHO TRI THỨC V2 (MẪU TỪ SEED {first_seed}):")
print("=" * 78)

for idx, sample in enumerate(all_seed_samples[first_seed][:3]):
    print(f"\n[Ảnh #{idx+1} | Image ID: {sample['image_id']}]")
    print("👉 Ground Truth (Tham chiếu thực tế):")
    for ref in sample["gts"][:2]:
        print(f'   - "{ref}"')
    print("🔍 Top Tri thức truy hồi & lọc ngữ nghĩa từ Kho V2:")
    for j, (c, rid) in enumerate(zip(sample["retrieved_caps"][:3], sample["retrieved_row_ids"][:3])):
        print(f'   + [KB #{j+1} | row={rid}] "{c}"')
        try:
            sem = describe_semantic(fetch_semantic(rid))
            if sem["objects"]:    print(f"      * Thực thể (Objects): {sem['objects']}")
            if sem["attributes"]: print(f"      * Thuộc tính (Attr) : {sem['attributes']}")
            if sem["actions"]:    print(f"      * Hành động (Act)   : {sem['actions']}")
            if sem["relations"]:  print(f"      * Quan hệ (SPO)     : {sem['relations']}")
        except Exception:
            pass
    gate_str = f" | Gate g={sample['gate_value']:.4f}" if sample.get("gate_value") is not None else ""
    print(f"✨ Caption Level 2 sinh ra (Seed {first_seed}{gate_str}):")
    print(f'   🎯 "{sample["hyp"]}"')
    print("-" * 78)


 MINH CHỨNG ĐỊNH TÍNH LEVEL 2 VỚI KHO TRI THỨC V2 (MẪU TỪ SEED 42):

[Ảnh #1 | Image ID: 42]
👉 Ground Truth (Tham chiếu thực tế):
   - "This wire metal rack holds several pairs of shoes and sandals"
   - "A dog sleeping on a show rack in the shoes."
🔍 Top Tri thức truy hồi & lọc ngữ nghĩa từ Kho V2:
   + [KB #1 | row=549788] "A dog laying against two pairs of shoes."
      * Thực thể (Objects): ['dog', 'pair', 'shoe']
      * Hành động (Act)   : [('dog', 'lay')]
      * Quan hệ (SPO)     : [('dog', 'against', 'pair'), ('pair', 'of', 'shoe')]
   + [KB #2 | row=549787] "A dog who is cuddled up nest to a pair of shoes."
      * Thực thể (Objects): ['dog', 'nest', 'pair', 'shoe']
      * Quan hệ (SPO)     : [('pair', 'of', 'shoe')]
   + [KB #3 | row=170529] "A sleeping dog with his face laying on a high top tennis shoe"
      * Thực thể (Objects): ['dog', 'face', 'tennis', 'tennis shoe']
      * Thuộc tính (Attr) : [('tennis shoe', 'high'), ('tennis shoe', 'top')]
      * Hành động (Act)  